# Static and dynamic sensitivity analysis

**Public-release note.** This notebook is part of a GitHub-ready version of the KMC memristor project. Notebook outputs were stripped to keep the repository lightweight, and path settings were adjusted to use repository-relative locations where needed.

**Purpose:** Runs one-at-a-time sensitivity analysis around the baseline parameter set for both the static and dynamic KMC workflows.

**Main manuscript role:** Used for main sensitivity figure generation and the supporting robustness tables in the Supplementary Information.

**Default assumption:** run the notebook from inside this repository so that the `results/` directory can be discovered automatically.


# KMC 静态 + 动态敏感性分析

这个 notebook **直接调用** 你放在：

- `notebooks/02_phase2_field_temperature_scan.ipynb`
- `notebooks/03_dynamic_feedback_validation.ipynb`

这两个文件中的**主代码单元**，然后做：

1. **静态 OAT 敏感性分析**  
   代表点：`m=0.25/0.90`, `T=700 K`, `E/E50≈0.9,1.0,1.1`  
   参数：  
   `SIGMA_DIS_EV`, `SIGMA_RES_EV`, `P_SIGMA`, `LC_BETA_CELLS`, `LC0_CELLS`,  
   `MU_X_EXTRA_EV`, `ALPHA_FIELD`, `ACTIVE_Z_MAX_FRAC`, `MU_SHIFT_EV`, `MU_SCALE`

2. **动态敏感性分析**  
   同样的 6 个代表点，比较 `static` vs `dynamic`  
   参数：  
   `DYN_UPDATE_EVENTS`, `DYN_ALPHA_SIGMA`, `DYN_M_COUNT_MODE`, `ACTIVE_Z_MAX_FRAC`

## 推荐运行顺序
先跑到 **“0) 配置”** 和 **“1) 加载工具函数”**，确认路径没问题。  
然后先跑 **静态敏感性**，最后再跑 **动态敏感性**。


In [ ]:
# ==============================
# 0) Repository-aware configuration
# ==============================
from pathlib import Path
import os, re, math, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nbformat

sns.set_theme(style="whitegrid", context="talk", font_scale=1.0)
warnings.filterwarnings("ignore")


def find_repo_root(start=None):
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    for path in [start] + list(start.parents):
        if (path / "README.md").exists() and (path / "notebooks").exists() and (path / "results").exists():
            return path
    return start

REPO_ROOT = find_repo_root()
NOTEBOOK_DIR = REPO_ROOT / "notebooks"
STATIC_NOTEBOOK = NOTEBOOK_DIR / "02_phase2_field_temperature_scan.ipynb"
DYNAMIC_NOTEBOOK = NOTEBOOK_DIR / "03_dynamic_feedback_validation.ipynb"

OUT_DIR = REPO_ROOT / "results" / "sensitivity"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------- Run preset ----------
RUN_PRESET = "quick"   # "quick" | "full"


In [ ]:

# ==============================
# 0b) 论文风格作图工具
# ==============================
import textwrap
from matplotlib.ticker import MaxNLocator

sns.set_theme(style="ticks", context="paper")
plt.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 600,
    "axes.titlesize": 15,
    "axes.titleweight": "bold",
    "axes.labelsize": 13,
    "axes.linewidth": 1.1,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.width": 1.0,
    "ytick.major.width": 1.0,
    "xtick.major.size": 4.5,
    "ytick.major.size": 4.5,
    "legend.fontsize": 10,
    "legend.title_fontsize": 10,
    "lines.linewidth": 2.0,
    "lines.markersize": 6.0,
    "font.family": "DejaVu Sans",
    "mathtext.default": "regular",
})

PARAM_LABELS = {
    "SIGMA_DIS_EV": r"Disorder barrier width, $\sigma_{dis}$ (eV)",
    "SIGMA_RES_EV": r"Residual barrier spread, $\sigma_{res}$ (eV)",
    "P_SIGMA": r"Barrier-disorder strength, $p_{\sigma}$",
    "LC_BETA_CELLS": r"Correlation length slope, $\beta_{LC}$ (cells)",
    "LC0_CELLS": r"Base correlation length, $LC_0$ (cells)",
    "MU_X_EXTRA_EV": r"Lateral migration penalty, $\mu_x^{extra}$ (eV)",
    "ALPHA_FIELD": r"Field-coupling factor, $\alpha_E$",
    "ACTIVE_Z_MAX_FRAC": r"Active-layer thickness fraction, $f_{active}$",
    "MU_SHIFT_EV": r"Depth-barrier offset, $\Delta \mu_z$ (eV)",
    "MU_SCALE": r"Depth-barrier scaling, $s_{\mu_z}$",
    "DYN_UPDATE_EVENTS": r"Dynamic update interval, $N_{update}$ (events)",
    "DYN_ALPHA_SIGMA": r"Dynamic disorder-response factor, $\alpha_{dyn}$",
    "DYN_M_COUNT_MODE": r"Dynamic order counting mode",
    "BASELINE": "Baseline",
}

METRIC_LABELS = {
    "formed_prob_mean6": r"Formation probability, $P_{form}$",
    "log10_t_set_med_mean6": r"Median switching time, $\log_{10}(t_{set}/s)$",
    "branches_crit_med_mean6": r"Critical branch count, $N_{branch}^{crit}$",
    "tortuosity_crit_med_mean6": r"Critical tortuosity, $\tau_{crit}$",
    "E50_m025": r"Threshold field, $E_{50}$ at $m_0=0.25$ (V nm$^{-1}$)",
    "E50_m090": r"Threshold field, $E_{50}$ at $m_0=0.90$ (V nm$^{-1}$)",
    "R2_Psi_branches": r"Correlation quality for $\Psi$ vs. branch count, $R^2$",
    "R2_Psi_tortuosity": r"Correlation quality for $\Psi$ vs. tortuosity, $R^2$",
    "delta_formed_prob_mean6": r"Shift in formation probability, $\Delta P_{form}$",
    "delta_log10_t_set_median_formed_mean6": r"Shift in switching time, $\Delta \log_{10}(t_{set}/s)$",
    "delta_branches_crit_median_formed_mean6": r"Shift in critical branch count, $\Delta N_{branch}^{crit}$",
    "delta_tortuosity_nm_crit_median_formed_mean6": r"Shift in critical tortuosity, $\Delta \tau_{crit}$",
    "delta_m_final_mean_mean6": r"Shift in final order parameter, $\Delta m_{final}$",
    "delta_Delta_final_mean_mean6": r"Shift in final disorder index, $\Delta \Delta_{final}$",
    "abs_delta_formed_prob_mean6": r"Absolute shift in formation probability, $|\Delta P_{form}|$",
    "abs_delta_log10_t_set_median_formed_mean6": r"Absolute shift in switching time, $|\Delta \log_{10}(t_{set}/s)|$",
    "abs_delta_branches_crit_median_formed_mean6": r"Absolute shift in critical branch count, $|\Delta N_{branch}^{crit}|$",
}


def wrap_axis_label(text, width=28):
    return "\n".join(textwrap.wrap(str(text), width=width, break_long_words=False, break_on_hyphens=False))


def param_label(param):
    return PARAM_LABELS.get(param, param)


def metric_label(metric):
    return METRIC_LABELS.get(metric, metric)


def format_level_value(param, value):
    if isinstance(value, str):
        mode_map = {
            "ion_only": "ions only",
            "ion_plus_fil": "ions + filament",
        }
        return mode_map.get(value, value)

    numeric = float(value)
    if param in {"SIGMA_DIS_EV", "SIGMA_RES_EV", "MU_X_EXTRA_EV", "MU_SHIFT_EV"}:
        return f"{numeric:.2f} eV"
    if param in {"LC_BETA_CELLS", "LC0_CELLS"}:
        return f"{numeric:.1f}"
    if param == "ACTIVE_Z_MAX_FRAC":
        return f"{numeric:.2f}"
    if param in {"P_SIGMA", "ALPHA_FIELD", "MU_SCALE", "DYN_ALPHA_SIGMA"}:
        return f"{numeric:.2f}"
    if param == "DYN_UPDATE_EVENTS":
        return f"{int(numeric)}"
    return f"{numeric:.3g}"


def get_static_level_value(param, level_name):
    if param == "BASELINE":
        return "baseline"
    if level_name == "base":
        if param == "MU_SHIFT_EV":
            return 0.0
        if param == "MU_SCALE":
            return 1.0
        return BASELINE_STATIC.get(param, "baseline")
    idx = int(str(level_name).replace("lvl", ""))
    return STATIC_PARAM_LEVELS[param][idx]


def get_dynamic_level_value(param, level_name):
    if param == "BASELINE":
        return "baseline"
    if level_name == "base":
        return BASELINE_DYNAMIC.get(param, "baseline")
    idx = int(str(level_name).replace("lvl", ""))
    return DYNAMIC_PARAM_LEVELS[param][idx]


def build_metric_heatmap(plot_df, title, cbar_label, out_path, cmap="mako", figsize=(12, 6)):
    pretty_df = plot_df.copy()
    pretty_df.index = [wrap_axis_label(param_label(p), 24) for p in pretty_df.index]
    pretty_df.columns = [wrap_axis_label(metric_label(c), 22) for c in pretty_df.columns]

    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(
        pretty_df,
        annot=True,
        fmt=".2f",
        cmap=cmap,
        linewidths=0.8,
        linecolor="white",
        cbar_kws={"label": cbar_label, "shrink": 0.92},
        annot_kws={"fontsize": 10},
        ax=ax,
    )
    ax.set_title(title, pad=12)
    ax.set_xlabel("Output metric")
    ax.set_ylabel("Model parameter")
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center")
    plt.setp(ax.get_yticklabels(), rotation=0)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()


def build_ranking_barplot(df, value_col, title, xlabel, out_path, figsize=(8.6, 5.4)):
    plot_df = df[["param", value_col]].dropna().sort_values(value_col, ascending=False).copy()
    plot_df["param_pretty"] = [wrap_axis_label(param_label(p), 28) for p in plot_df["param"]]

    fig, ax = plt.subplots(figsize=figsize)
    sns.barplot(data=plot_df, y="param_pretty", x=value_col, ax=ax, color="0.35")
    ax.set_title(title, pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Model parameter")
    ax.grid(axis="x", alpha=0.25)
    ax.grid(axis="y", visible=False)

    xmax = float(plot_df[value_col].max()) if len(plot_df) else 1.0
    pad = 0.02 * xmax if xmax > 0 else 0.02
    for patch in ax.patches:
        width = patch.get_width()
        y = patch.get_y() + patch.get_height() / 2.0
        ax.text(width + pad, y, f"{width:.2f}", va="center", ha="left", fontsize=10)

    ax.set_xlim(0, xmax * 1.12 if xmax > 0 else 1.0)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()


def build_static_trajectory_panels(df, metric, out_path):
    params = [p for p in STATIC_PARAM_LEVELS.keys() if p in set(df["param"]) ]
    n = len(params)
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12.8, 2.95 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, param in zip(axes, params):
        sub = df[df["param"] == param].copy()
        order_vals = STATIC_PARAM_LEVELS[param]
        order_labels = [format_level_value(param, v) for v in order_vals]
        sub["level_value"] = sub["level_name"].map(lambda x: get_static_level_value(param, x))
        order_index = {repr(v): i for i, v in enumerate(order_vals)}
        sub["sort_idx"] = sub["level_value"].map(lambda v: order_index.get(repr(v), 999))
        sub = sub.sort_values("sort_idx")

        ax.plot(range(len(sub)), sub[metric].values, marker="o")
        ax.set_title(wrap_axis_label(param_label(param), 30), fontsize=12, pad=8)
        ax.set_xticks(range(len(order_labels)))
        ax.set_xticklabels(order_labels, rotation=0)
        ax.set_xlabel("Tested value")
        ax.set_ylabel(metric_label(metric))
        ax.grid(axis="y", alpha=0.25)
        ax.grid(axis="x", visible=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        baseline_val = BASELINE_STATIC.get(param, None)
        if param == "MU_SHIFT_EV":
            baseline_val = 0.0
        elif param == "MU_SCALE":
            baseline_val = 1.0
        if baseline_val in order_vals:
            base_idx = order_vals.index(baseline_val)
            ax.axvline(base_idx, ls="--", lw=1.0, color="0.55", alpha=0.85)

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(
        f"Static OAT response trajectories: {metric_label(metric)}",
        fontsize=16,
        fontweight="bold",
        y=1.01,
    )
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()


def build_dynamic_trajectory_panels(df, metric, out_path):
    params = [p for p in DYNAMIC_PARAM_LEVELS.keys() if p in set(df["param"]) ]
    n = len(params)
    ncols = 2
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(12.8, 3.1 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, param in zip(axes, params):
        sub = df[df["param"] == param].copy()
        order_vals = DYNAMIC_PARAM_LEVELS[param]
        order_labels = [format_level_value(param, v) for v in order_vals]
        sub["level_value"] = sub["level_name"].map(lambda x: get_dynamic_level_value(param, x))
        order_index = {repr(v): i for i, v in enumerate(order_vals)}
        sub["sort_idx"] = sub["level_value"].map(lambda v: order_index.get(repr(v), 999))
        sub = sub.sort_values("sort_idx")

        ax.plot(range(len(sub)), sub[metric].values, marker="o")
        ax.set_title(wrap_axis_label(param_label(param), 30), fontsize=12, pad=8)
        ax.set_xticks(range(len(order_labels)))
        ax.set_xticklabels(order_labels, rotation=0)
        ax.set_xlabel("Tested value")
        ax.set_ylabel(metric_label(metric))
        ax.grid(axis="y", alpha=0.25)
        ax.grid(axis="x", visible=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

        baseline_val = BASELINE_DYNAMIC.get(param, None)
        if baseline_val in order_vals:
            base_idx = order_vals.index(baseline_val)
            ax.axvline(base_idx, ls="--", lw=1.0, color="0.55", alpha=0.85)

    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(
        f"Dynamic sensitivity trajectories: {metric_label(metric)}",
        fontsize=16,
        fontweight="bold",
        y=1.02,
    )
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight")
    plt.show()


In [ ]:

# ==============================
# 1) 加载 notebook 的工具函数
# ==============================
def extract_main_code_cell(nb_path: Path) -> str:
    nb = nbformat.read(nb_path, as_version=4)
    for cell in nb.cells:
        if cell.cell_type == "code":
            return cell.source
    raise ValueError(f"没有找到代码单元: {nb_path}")

def py_expr(value):
    if isinstance(value, np.ndarray):
        dtype = "float" if value.dtype.kind in "fc" else "int"
        return f"np.array({value.tolist()}, dtype={dtype})"
    if isinstance(value, Path):
        return f"Path(r'{str(value)}')"
    if isinstance(value, str):
        return repr(value)
    if isinstance(value, bool):
        return "True" if value else "False"
    return repr(value)

def patch_notebook_source(src: str, overrides: dict) -> str:
    src = src.replace("cache=True", "cache=False")
    for key, value in overrides.items():
        expr = py_expr(value)
        pat = rf"(?m)^({re.escape(key)}\s*=\s*).*$"
        if re.search(pat, src):
            src = re.sub(pat, lambda m, expr=expr: m.group(1) + expr, src)
        else:
            src = f"{key} = {expr}\n" + src
    return src

def load_notebook_namespace(nb_path: Path, overrides: dict | None = None):
    src = extract_main_code_cell(nb_path)
    if overrides:
        src = patch_notebook_source(src, overrides)
    ns = {"__name__": "__main__"}
    exec(src, ns)
    return ns

def build_mu_variant(base_mu, shift_ev=0.0, scale=1.0):
    base_mu = np.asarray(base_mu, dtype=float)
    mu_mean = np.mean(base_mu)
    return mu_mean + scale * (base_mu - mu_mean) + shift_ev

def stable_case_name(param, level_name):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", f"{param}_{level_name}")

def maybe_add_crit_metrics(ns, out: dict):
    """
    动态 notebook 原始版可能没有 branches_crit/tortuosity_nm_crit。
    如果缺失，就从 snapshots['Critical'] 临时补。
    """
    if ("branches_crit" in out) and (out.get("branches_crit") is not None):
        return out
    snaps = out.get("snapshots", {})
    if "Critical" in snaps:
        occ_crit = snaps["Critical"]
        meander_rms_nm_crit, neck_nm_crit, branches_crit, comx_nm_crit, drift_flag_crit, tort_nm_crit = ns["morphology_from_occ"](occ_crit)
        out["meander_rms_nm_crit"] = float(meander_rms_nm_crit)
        out["neck_nm_crit"] = float(neck_nm_crit)
        out["branches_crit"] = int(branches_crit)
        out["comx_nm_crit"] = float(comx_nm_crit)
        out["drift_crit"] = int(drift_flag_crit)
        out["tortuosity_nm_crit"] = float(tort_nm_crit)
    return out

def print_namespace_core(ns, title="namespace"):
    keys = [
        "SIGMA_DIS_EV", "SIGMA_RES_EV", "P_SIGMA",
        "LC_BETA_CELLS", "LC0_CELLS",
        "MU_X_EXTRA_EV", "ALPHA_FIELD",
        "ACTIVE_Z_MAX_FRAC",
        "DYN_UPDATE_EVENTS", "DYN_ALPHA_SIGMA", "DYN_M_COUNT_MODE"
    ]
    print(f"\n[{title}] core parameters")
    for k in keys:
        if k in ns:
            print(f"{k:>18s} = {ns[k]}")


In [ ]:

# ==============================
# 2) 运行与汇总函数
# ==============================
def make_static_overrides(param=None, level=None, save_subdir="static_tmp"):
    overrides = dict(BASELINE_STATIC)
    overrides["SAVE_DIR"] = OUT_DIR / save_subdir
    if param is None:
        return overrides
    if param == "MU_SHIFT_EV":
        overrides["MU_DFT_Z_EV"] = build_mu_variant(BASELINE_STATIC["MU_DFT_Z_EV"], shift_ev=float(level), scale=1.0)
    elif param == "MU_SCALE":
        overrides["MU_DFT_Z_EV"] = build_mu_variant(BASELINE_STATIC["MU_DFT_Z_EV"], shift_ev=0.0, scale=float(level))
    else:
        overrides[param] = level
    return overrides

def make_dynamic_overrides(param=None, level=None, save_subdir="dynamic_tmp"):
    overrides = dict(BASELINE_DYNAMIC)
    overrides["SAVE_DIR"] = OUT_DIR / save_subdir
    if param is not None:
        overrides[param] = level
    return overrides

def run_static_points(ns, points, n_structure_seeds=2, n_mc_per_structure=2):
    rows = []
    for p in points:
        for sdev in range(n_structure_seeds):
            structure_seed = 2025 + 1000 * sdev + int(round(p["m0"] * 100)) + int(round(p["E"] * 1000))
            for smc in range(n_mc_per_structure):
                mc_seed = 10_000 * structure_seed + smc
                out = ns["simulate_one_phase2"](
                    m0_target=p["m0"], E_vnm=p["E"], T=p["T"],
                    structure_seed=structure_seed, mc_seed=mc_seed,
                    return_snapshots=False, store_rate_samples=False
                )
                out["label"] = p["label"]
                out["E_over_E50_target"] = p.get("E_over_E50_target", np.nan)
                rows.append(out)
    return pd.DataFrame(rows)

def run_static_E50_sweeps(ns, sweep_map, T_fixed=700.0, n_structure_seeds=2, n_mc_per_structure=2):
    rows = []
    for m0, e_list in sweep_map.items():
        for E0 in e_list:
            for sdev in range(n_structure_seeds):
                structure_seed = 3031 + 1000 * sdev + int(round(m0 * 100)) + int(round(E0 * 1000))
                for smc in range(n_mc_per_structure):
                    mc_seed = 20_000 * structure_seed + smc
                    out = ns["simulate_one_phase2"](
                        m0_target=float(m0), E_vnm=float(E0), T=float(T_fixed),
                        structure_seed=structure_seed, mc_seed=mc_seed,
                        return_snapshots=False, store_rate_samples=False
                    )
                    out["sweep_type"] = "E50"
                    rows.append(out)
    return pd.DataFrame(rows)

def run_dynamic_points(ns, points, n_structure_seeds=2, n_mc_per_structure=2):
    rows = []
    for p in points:
        for mode in ["static", "dynamic"]:
            for sdev in range(n_structure_seeds):
                structure_seed = 4041 + 1000 * sdev + int(round(p["m0"] * 100)) + int(round(p["E"] * 1000))
                for smc in range(n_mc_per_structure):
                    mc_seed = 30_000 * structure_seed + smc
                    out = ns["simulate_one_static_or_dynamic"](
                        m0_target=p["m0"], E_vnm=p["E"], T=p["T"],
                        structure_seed=structure_seed, mc_seed=mc_seed,
                        dynamic=(mode == "dynamic"),
                        return_snapshots=DYNAMIC_RETURN_SNAPSHOTS,
                        store_rate_samples=False,
                        dynamic_count_mode=ns.get("DYN_M_COUNT_MODE", "ion_plus_fil"),
                        dynamic_update_events=ns.get("DYN_UPDATE_EVENTS", 100),
                        dynamic_alpha_sigma=ns.get("DYN_ALPHA_SIGMA", 0.10),
                        dynamic_alpha_lc=ns.get("DYN_ALPHA_LC", 0.10),
                        dynamic_update_mode=ns.get("DYN_UPDATE_MODE", "sigma_only"),
                        dynamic_eta_mix_gamma=ns.get("DYN_ETA_MIX_GAMMA", 0.15),
                    )
                    out = maybe_add_crit_metrics(ns, out)
                    out["mode"] = mode
                    out["label"] = p["label"]
                    out["E_over_E50_target"] = p.get("E_over_E50_target", np.nan)
                    rows.append(out)
    return pd.DataFrame(rows)

def summarize_runs(df, group_cols=("label", "m_target", "E", "T")):
    rows = []
    for keys, g in df.groupby(list(group_cols), dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {k: v for k, v in zip(group_cols, keys)}
        row["n_runs"] = len(g)
        row["formed_prob"] = float(g["formed"].mean()) if "formed" in g.columns else np.nan
        gf = g[g["formed"] == 1].copy() if "formed" in g.columns else g.copy()
        row["formed_runs"] = len(gf)

        for col in ["sigma_E", "lc", "Delta", "m_actual", "m_final", "sigma_final", "Delta_final"]:
            if col in g.columns:
                row[f"{col}_mean"] = float(g[col].mean())

        if len(gf):
            if "t_set" in gf.columns:
                row["log10_t_set_median_formed"] = float(np.nanmedian(np.log10(gf["t_set"].values)))
                row["log10_t_set_mean_formed"] = float(np.nanmean(np.log10(gf["t_set"].values)))
            for col in ["branches_crit", "tortuosity_nm_crit", "neck_nm_crit",
                        "branches", "tortuosity_nm", "neck_nm"]:
                if col in gf.columns:
                    row[f"{col}_median_formed"] = float(np.nanmedian(gf[col].values))
                    row[f"{col}_mean_formed"] = float(np.nanmean(gf[col].values))
        else:
            row["log10_t_set_median_formed"] = np.nan
            row["log10_t_set_mean_formed"] = np.nan

        rows.append(row)
    return pd.DataFrame(rows)

def fit_E50_from_summary(summary_df):
    rows = []
    for m0, g in summary_df.groupby("m_target"):
        sub = g[["E", "formed_prob"]].dropna().sort_values("E")
        if len(sub) < 3:
            rows.append({"m_target": m0, "fit_ok": False, "E50": np.nan, "wE": np.nan, "a": np.nan, "b": np.nan})
            continue
        E = sub["E"].values.astype(float)
        p = np.clip(sub["formed_prob"].values.astype(float), 1e-3, 1 - 1e-3)
        y = np.log(p / (1 - p))
        A = np.column_stack([np.ones_like(E), E])
        coef, *_ = np.linalg.lstsq(A, y, rcond=None)
        a, b = coef
        if abs(b) < 1e-10:
            rows.append({"m_target": m0, "fit_ok": False, "E50": np.nan, "wE": np.nan, "a": a, "b": b})
        else:
            rows.append({
                "m_target": float(m0), "fit_ok": True,
                "E50": float(-a / b), "wE": float(1 / abs(b)),
                "a": float(a), "b": float(b),
            })
    return pd.DataFrame(rows)

def attach_E50_and_Psi(point_summary, e50_table):
    out = point_summary.merge(e50_table[["m_target", "E50", "wE", "fit_ok"]], on="m_target", how="left")
    out["E_norm"] = out["E"] / out["E50"]
    out["E_hat"] = (out["E"] - out["E50"]) / out["wE"]
    out["Psi_simple"] = out["E_norm"] / out["Delta_mean"]
    return out

def fit_quadratic_r2(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    x, y = x[mask], y[mask]
    if len(x) < 4:
        return np.nan
    X = np.column_stack([np.ones_like(x), x, x**2])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coef
    ss_res = np.sum((y - y_pred)**2)
    ss_tot = np.sum((y - np.mean(y))**2) + 1e-12
    return float(1 - ss_res / ss_tot)

def aggregate_case_metrics(point_summary, e50_table):
    out = {}
    out["formed_prob_mean6"] = float(np.nanmean(point_summary["formed_prob"]))
    out["log10_t_set_med_mean6"] = float(np.nanmean(point_summary["log10_t_set_median_formed"]))
    if "branches_crit_median_formed" in point_summary.columns:
        out["branches_crit_med_mean6"] = float(np.nanmean(point_summary["branches_crit_median_formed"]))
    if "tortuosity_nm_crit_median_formed" in point_summary.columns:
        out["tortuosity_crit_med_mean6"] = float(np.nanmean(point_summary["tortuosity_nm_crit_median_formed"]))
    for m0 in [0.25, 0.90]:
        sub = e50_table[e50_table["m_target"].round(2) == round(m0, 2)]
        out[f"E50_m{int(m0*100):03d}"] = float(sub["E50"].iloc[0]) if len(sub) else np.nan
    if "branches_crit_median_formed" in point_summary.columns:
        out["R2_Psi_branches"] = fit_quadratic_r2(point_summary["Psi_simple"], point_summary["branches_crit_median_formed"])
    else:
        out["R2_Psi_branches"] = np.nan
    if "tortuosity_nm_crit_median_formed" in point_summary.columns:
        out["R2_Psi_tortuosity"] = fit_quadratic_r2(point_summary["Psi_simple"], point_summary["tortuosity_nm_crit_median_formed"])
    else:
        out["R2_Psi_tortuosity"] = np.nan
    return out

def aggregate_dynamic_case_metrics(point_summary):
    out = {}
    pivot_cols = [
        "formed_prob",
        "log10_t_set_median_formed",
        "branches_crit_median_formed",
        "tortuosity_nm_crit_median_formed",
        "m_final_mean",
        "Delta_final_mean",
    ]
    ps = point_summary.copy()
    for col in pivot_cols:
        if col not in ps.columns:
            ps[col] = np.nan

    piv = ps.pivot_table(
        index=["label", "m_target", "E", "T"],
        columns="mode",
        values=pivot_cols,
        aggfunc="first"
    )
    piv.columns = [f"{a}__{b}" for a, b in piv.columns]
    piv = piv.reset_index()

    for base_col in pivot_cols:
        s_col = f"{base_col}__static"
        d_col = f"{base_col}__dynamic"
        if s_col in piv.columns and d_col in piv.columns:
            out[f"{base_col}_static_mean6"] = float(np.nanmean(piv[s_col]))
            out[f"{base_col}_dynamic_mean6"] = float(np.nanmean(piv[d_col]))
            out[f"delta_{base_col}_mean6"] = float(np.nanmean(piv[d_col] - piv[s_col]))
            out[f"abs_delta_{base_col}_mean6"] = float(np.nanmean(np.abs(piv[d_col] - piv[s_col])))
    return out

def compute_param_impact(case_metrics_df, metric_cols, baseline_level_name="base"):
    rows = []
    for param, g in case_metrics_df.groupby("param"):
        base = g[g["level_name"] == baseline_level_name]
        if len(base) == 0:
            continue
        base = base.iloc[0]
        row = {"param": param}
        for m in metric_cols:
            vals = g[m].values.astype(float)
            base_val = float(base[m])
            row[m] = (np.nanmax(vals) - np.nanmin(vals)) / (abs(base_val) + 1e-9) if np.any(np.isfinite(vals)) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def show_head(title, df, n=10):
    print("\n" + "="*80)
    print(title)
    print(df.head(n))


In [ ]:

# ==============================
# 3) 静态敏感性分析（OAT）
# ==============================
def run_static_sensitivity(static_notebook=STATIC_NOTEBOOK):
    all_raw_points = []
    all_case_metrics = []
    all_e50_tables = []

    static_cases = [("BASELINE", "base", None, None)]
    for param, levels in STATIC_PARAM_LEVELS.items():
        for i, level in enumerate(levels):
            level_name = "base" if (param not in ["MU_SHIFT_EV", "MU_SCALE"] and level == BASELINE_STATIC.get(param, object())) else f"lvl{i}"
            static_cases.append((param, level_name, param, level))

    for case_idx, (param_name, level_name, override_key, override_value) in enumerate(static_cases, start=1):
        save_subdir = f"static_{case_idx:02d}_{stable_case_name(param_name, level_name)}"
        overrides = make_static_overrides(param=override_key, level=override_value, save_subdir=save_subdir)
        ns = load_notebook_namespace(static_notebook, overrides)
        print_namespace_core(ns, title=f"static case {case_idx}: {param_name}/{level_name}")

        df_points_raw = run_static_points(
            ns, STATIC_REP_POINTS,
            n_structure_seeds=STATIC_N_STRUCTURE_SEEDS,
            n_mc_per_structure=STATIC_N_MC_PER_STRUCTURE
        )
        df_points_raw["param"] = param_name
        df_points_raw["level_name"] = level_name
        all_raw_points.append(df_points_raw)

        point_summary = summarize_runs(df_points_raw, group_cols=("label", "m_target", "E", "T"))
        point_summary["param"] = param_name
        point_summary["level_name"] = level_name

        df_e50_raw = run_static_E50_sweeps(
            ns, STATIC_E50_SWEEPS,
            T_fixed=700.0,
            n_structure_seeds=STATIC_N_STRUCTURE_SEEDS,
            n_mc_per_structure=STATIC_N_MC_PER_STRUCTURE
        )
        e50_summary = summarize_runs(df_e50_raw, group_cols=("m_target", "E", "T"))
        e50_table = fit_E50_from_summary(e50_summary)
        e50_table["param"] = param_name
        e50_table["level_name"] = level_name
        all_e50_tables.append(e50_table)

        point_summary = attach_E50_and_Psi(point_summary, e50_table)
        case_metrics = aggregate_case_metrics(point_summary, e50_table)
        case_metrics["param"] = param_name
        case_metrics["level_name"] = level_name
        all_case_metrics.append(case_metrics)

    raw_points_df = pd.concat(all_raw_points, ignore_index=True)
    case_metrics_df = pd.DataFrame(all_case_metrics)
    e50_tables_df = pd.concat(all_e50_tables, ignore_index=True)

    raw_points_df.to_csv(OUT_DIR / "static_oat_points_raw.csv", index=False)
    case_metrics_df.to_csv(OUT_DIR / "static_oat_case_metrics.csv", index=False)
    e50_tables_df.to_csv(OUT_DIR / "static_oat_e50_tables.csv", index=False)

    show_head("static_oat_case_metrics", case_metrics_df)
    show_head("static_oat_e50_tables", e50_tables_df)
    return raw_points_df, case_metrics_df, e50_tables_df

static_raw_points_df, static_case_metrics_df, static_e50_tables_df = run_static_sensitivity()


In [ ]:

# ==============================
# 4) 静态敏感性：汇总 + 论文风格图
# ==============================
STATIC_METRIC_COLS = [
    "formed_prob_mean6",
    "log10_t_set_med_mean6",
    "branches_crit_med_mean6",
    "tortuosity_crit_med_mean6",
    "E50_m025",
    "E50_m090",
    "R2_Psi_branches",
    "R2_Psi_tortuosity",
]

static_impact_df = compute_param_impact(static_case_metrics_df, STATIC_METRIC_COLS, baseline_level_name="base")
static_impact_df.to_csv(OUT_DIR / "static_oat_param_impact.csv", index=False)
show_head("static_oat_param_impact", static_impact_df)

plot_df = static_impact_df.set_index("param")[STATIC_METRIC_COLS]
build_metric_heatmap(
    plot_df,
    title="Static OAT sensitivity map of key KMC outputs\nNormalized impact relative to baseline (range / baseline)",
    cbar_label="Normalized impact, range / baseline",
    out_path=OUT_DIR / "static_oat_impact_heatmap.png",
    cmap="mako",
    figsize=(13.2, 6.8),
)

build_ranking_barplot(
    static_impact_df,
    value_col="R2_Psi_branches",
    title=r"Static parameter ranking by impact on the $\Psi$-branch correlation",
    xlabel=r"Normalized impact on $R^2$ for the $\Psi$-branch relation",
    out_path=OUT_DIR / "static_oat_r2psi_branches_ranking.png",
    figsize=(8.8, 5.6),
)

for metric in ["formed_prob_mean6", "branches_crit_med_mean6", "E50_m025", "E50_m090"]:
    build_static_trajectory_panels(
        static_case_metrics_df,
        metric=metric,
        out_path=OUT_DIR / f"static_oat_rawtraj_{metric}.png",
    )


In [ ]:

# ==============================
# 5) 动态敏感性分析
# ==============================
def run_dynamic_sensitivity(dynamic_notebook=DYNAMIC_NOTEBOOK):
    all_raw = []
    all_case_metrics = []

    dynamic_cases = [("BASELINE", "base", None, None)]
    for param, levels in DYNAMIC_PARAM_LEVELS.items():
        for i, level in enumerate(levels):
            level_name = "base" if (param in BASELINE_DYNAMIC and level == BASELINE_DYNAMIC[param]) else f"lvl{i}"
            dynamic_cases.append((param, level_name, param, level))

    for case_idx, (param_name, level_name, override_key, override_value) in enumerate(dynamic_cases, start=1):
        save_subdir = f"dynamic_{case_idx:02d}_{stable_case_name(param_name, level_name)}"
        overrides = make_dynamic_overrides(param=override_key, level=override_value, save_subdir=save_subdir)
        ns = load_notebook_namespace(dynamic_notebook, overrides)
        print_namespace_core(ns, title=f"dynamic case {case_idx}: {param_name}/{level_name}")

        df_raw = run_dynamic_points(
            ns, STATIC_REP_POINTS,
            n_structure_seeds=DYNAMIC_N_STRUCTURE_SEEDS,
            n_mc_per_structure=DYNAMIC_N_MC_PER_STRUCTURE
        )
        df_raw["param"] = param_name
        df_raw["level_name"] = level_name
        all_raw.append(df_raw)

        point_summary = summarize_runs(df_raw, group_cols=("label", "mode", "m_target", "E", "T"))
        case_metrics = aggregate_dynamic_case_metrics(point_summary)
        case_metrics["param"] = param_name
        case_metrics["level_name"] = level_name
        all_case_metrics.append(case_metrics)

    raw_df = pd.concat(all_raw, ignore_index=True)
    case_metrics_df = pd.DataFrame(all_case_metrics)

    raw_df.to_csv(OUT_DIR / "dynamic_sens_points_raw.csv", index=False)
    case_metrics_df.to_csv(OUT_DIR / "dynamic_sens_case_metrics.csv", index=False)

    show_head("dynamic_sens_case_metrics", case_metrics_df)
    return raw_df, case_metrics_df

dynamic_raw_df, dynamic_case_metrics_df = run_dynamic_sensitivity()


In [ ]:

# ==============================
# 6) 动态敏感性：汇总 + 论文风格图
# ==============================
DYNAMIC_METRIC_COLS = [
    "delta_formed_prob_mean6",
    "delta_log10_t_set_median_formed_mean6",
    "delta_branches_crit_median_formed_mean6",
    "delta_tortuosity_nm_crit_median_formed_mean6",
    "delta_m_final_mean_mean6",
    "delta_Delta_final_mean_mean6",
    "abs_delta_formed_prob_mean6",
    "abs_delta_log10_t_set_median_formed_mean6",
    "abs_delta_branches_crit_median_formed_mean6",
]

dynamic_impact_df = compute_param_impact(dynamic_case_metrics_df, DYNAMIC_METRIC_COLS, baseline_level_name="base")
dynamic_impact_df.to_csv(OUT_DIR / "dynamic_param_impact.csv", index=False)
show_head("dynamic_param_impact", dynamic_impact_df)

plot_df = dynamic_impact_df.set_index("param")[DYNAMIC_METRIC_COLS]
build_metric_heatmap(
    plot_df,
    title="Dynamic sensitivity map of static-to-dynamic response shifts\nNormalized impact relative to baseline (range / baseline)",
    cbar_label="Normalized impact, range / baseline",
    out_path=OUT_DIR / "dynamic_impact_heatmap.png",
    cmap="rocket_r",
    figsize=(14.0, 5.9),
)

build_ranking_barplot(
    dynamic_impact_df,
    value_col="abs_delta_branches_crit_median_formed_mean6",
    title="Dynamic parameter ranking by impact on the critical branch-count shift",
    xlabel=r"Normalized impact on $|\Delta N_{branch}^{crit}|$",
    out_path=OUT_DIR / "dynamic_branches_shift_ranking.png",
    figsize=(8.8, 5.4),
)

for metric in [
    "delta_formed_prob_mean6",
    "delta_branches_crit_median_formed_mean6",
    "abs_delta_branches_crit_median_formed_mean6",
]:
    build_dynamic_trajectory_panels(
        dynamic_case_metrics_df,
        metric=metric,
        out_path=OUT_DIR / f"dynamic_traj_{metric}.png",
    )


## 7) 你应该改哪里

### A. 路径
只要改：
- `BASE_DIR`

### B. 静态代表点
只改：
- `STATIC_REP_POINTS`
- `STATIC_E50_SWEEPS`

### C. 静态敏感性参数
只改：
- `STATIC_PARAM_LEVELS`

如果你只想先做精简版，先留这 5 个：
- `SIGMA_DIS_EV`
- `P_SIGMA`
- `MU_X_EXTRA_EV`
- `ALPHA_FIELD`
- `ACTIVE_Z_MAX_FRAC`

### D. 动态敏感性参数
只改：
- `DYNAMIC_PARAM_LEVELS`

如果你只想先做精简版，先留这 4 个：
- `DYN_UPDATE_EVENTS`
- `DYN_ALPHA_SIGMA`
- `DYN_M_COUNT_MODE`
- `ACTIVE_Z_MAX_FRAC`

### E. 统计精度
只改：
- `RUN_PRESET`
- `STATIC_N_STRUCTURE_SEEDS`
- `STATIC_N_MC_PER_STRUCTURE`
- `DYNAMIC_N_STRUCTURE_SEEDS`
- `DYNAMIC_N_MC_PER_STRUCTURE`

---

## 8) 你跑完后先看什么

### 静态先看
- `static_oat_param_impact.csv`
- `static_oat_impact_heatmap.png`
- `static_oat_r2psi_branches_ranking.png`

### 动态先看
- `dynamic_param_impact.csv`
- `dynamic_impact_heatmap.png`
- `dynamic_branches_shift_ranking.png`

---

## 9) 如何解读
如果静态里：
- `R2_Psi_branches` 对某几个参数特别敏感  
说明这些参数是你后面最该优先物理校准的。

如果动态里：
- `ACTIVE_Z_MAX_FRAC`
- `DYN_ALPHA_SIGMA`
- `DYN_UPDATE_EVENTS`

对 `delta_branches_crit_median_formed_mean6` 最敏感  
说明 dynamic m 的主信号是真实的，不只是噪声。


In [ ]:
# =========================
# Figure 6. Sensitivity and dynamic validation
# 直接接在你当前 notebook 最后面运行
# =========================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- 基本检查 ----------
required_vars = [
    "OUT_DIR",
    "STATIC_REP_POINTS",
    "static_impact_df",
    "dynamic_impact_df",
    "dynamic_raw_df",
    "summarize_runs",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"缺少这些变量，说明前面的单元还没跑完：{missing}")

PAPER_DIR = OUT_DIR / "paper_figures"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 论文风格 ----------
sns.set_theme(style="whitegrid", context="talk", font_scale=1.0)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300

# ---------- 面板可调 ----------
# panel (a) 里保留最核心的 static 指标，避免太挤
STATIC_PANEL_A_COLS = [
    "formed_prob_mean6",
    "log10_t_set_med_mean6",
    "branches_crit_med_mean6",
    "tortuosity_crit_med_mean6",
    "E50_m025",
    "E50_m090",
    "R2_Psi_branches",
]

# panel (c) 里保留最核心的 dynamic 指标
DYNAMIC_PANEL_C_COLS = [
    "abs_delta_formed_prob_mean6",
    "abs_delta_log10_t_set_median_formed_mean6",
    "abs_delta_branches_crit_median_formed_mean6",
    "delta_m_final_mean_mean6",
    "delta_Delta_final_mean_mean6",
]

# panel (d) 想比较哪个代表性指标
# 推荐用 branches_crit_median_formed，因为和你 panel (b) 主线一致
# 如果你想看 dynamic m 本身，也可以改成 "m_final_mean"
PANEL_D_METRIC = "branches_crit_median_formed"

metric_label_map = {
    "formed_prob_mean6": "Formed prob.\n(mean)",
    "log10_t_set_med_mean6": "log10(tset)\n(median)",
    "branches_crit_med_mean6": "Branches@crit\n(median)",
    "tortuosity_crit_med_mean6": "Tortuosity@crit\n(median)",
    "E50_m025": "E50 (m=0.25)",
    "E50_m090": "E50 (m=0.90)",
    "R2_Psi_branches": "R²(Psi, branches)",

    "abs_delta_formed_prob_mean6": "|Δ formed prob.|",
    "abs_delta_log10_t_set_median_formed_mean6": "|Δ log10(tset)|",
    "abs_delta_branches_crit_median_formed_mean6": "|Δ branches@crit|",
    "delta_m_final_mean_mean6": "Δ m_final",
    "delta_Delta_final_mean_mean6": "Δ Delta_final",

    "branches_crit_median_formed": "Branches at critical state",
    "tortuosity_nm_crit_median_formed": "Tortuosity at critical state (nm)",
    "m_final_mean": "Final m (mean)",
    "formed_prob": "Formed probability",
    "log10_t_set_median_formed": "log10(tset) (formed)",
}

param_label_map = {
    "SIGMA_DIS_EV": r"$\sigma_{\mathrm{dis}}$",
    "SIGMA_RES_EV": r"$\sigma_{\mathrm{res}}$",
    "P_SIGMA": r"$p_{\sigma}$",
    "LC_BETA_CELLS": r"$\beta_{\mathrm{lc}}$",
    "LC0_CELLS": r"$lc_0$",
    "MU_X_EXTRA_EV": r"$\mu_x^{extra}$",
    "ALPHA_FIELD": r"$\alpha_E$",
    "ACTIVE_Z_MAX_FRAC": r"$z_{active}^{max}$",
    "MU_SHIFT_EV": r"$\Delta \mu_z$",
    "MU_SCALE": r"$s(\mu_z)$",
    "DYN_UPDATE_EVENTS": "Dyn. update\ninterval",
    "DYN_ALPHA_SIGMA": r"Dyn. $\alpha_{\sigma}$",
    "DYN_M_COUNT_MODE": "Dyn. m-count\nmode",
    "BASELINE": "Baseline",
}

def _pretty_param_name(x):
    return param_label_map.get(x, str(x).replace("_", "\n"))

def _pretty_metric_name(x):
    return metric_label_map.get(x, x)

def _pretty_point_label(lbl):
    # m025_sub -> m=0.25\nsub
    lbl = str(lbl)
    lbl = lbl.replace("m025", "m=0.25")
    lbl = lbl.replace("m090", "m=0.90")
    lbl = lbl.replace("_sub", "\nsub")
    lbl = lbl.replace("_thr", "\nthr")
    lbl = lbl.replace("_sup", "\nsup")
    return lbl

# ---------- panel (a): static heatmap ----------
static_plot_df = static_impact_df.copy()
static_plot_df = static_plot_df[static_plot_df["param"] != "BASELINE"].copy()
static_cols_use = [c for c in STATIC_PANEL_A_COLS if c in static_plot_df.columns]

static_plot_df = (
    static_plot_df.set_index("param")[static_cols_use]
    .rename(index=_pretty_param_name)
    .rename(columns=_pretty_metric_name)
)

# ---------- panel (b): static ranking ----------
rank_col = "R2_Psi_branches"
rank_df = static_impact_df.copy()
rank_df = rank_df[(rank_df["param"] != "BASELINE") & rank_df[rank_col].notna()].copy()
rank_df["param_show"] = rank_df["param"].map(_pretty_param_name)
rank_df = rank_df.sort_values(rank_col, ascending=False)

# ---------- panel (c): dynamic heatmap ----------
dynamic_plot_df = dynamic_impact_df.copy()
dynamic_plot_df = dynamic_plot_df[dynamic_plot_df["param"] != "BASELINE"].copy()
dynamic_cols_use = [c for c in DYNAMIC_PANEL_C_COLS if c in dynamic_plot_df.columns]

dynamic_plot_df = (
    dynamic_plot_df.set_index("param")[dynamic_cols_use]
    .rename(index=_pretty_param_name)
    .rename(columns=_pretty_metric_name)
)

# ---------- panel (d): baseline representative-point comparison ----------
base_dyn = dynamic_raw_df.copy()
if "param" in base_dyn.columns:
    base_dyn = base_dyn[base_dyn["param"] == "BASELINE"].copy()
if "level_name" in base_dyn.columns:
    base_dyn = base_dyn[base_dyn["level_name"] == "base"].copy()

point_summary_d = summarize_runs(
    base_dyn,
    group_cols=("label", "mode", "m_target", "E", "T")
).copy()

if PANEL_D_METRIC not in point_summary_d.columns:
    fallback_candidates = [
        "branches_crit_median_formed",
        "m_final_mean",
        "formed_prob",
        "log10_t_set_median_formed",
    ]
    fallback = next((c for c in fallback_candidates if c in point_summary_d.columns), None)
    if fallback is None:
        raise RuntimeError(
            f"panel (d) 需要的指标 {PANEL_D_METRIC} 不存在，而且也没有找到合适的备选列。"
        )
    print(f"[panel d] 指标 {PANEL_D_METRIC} 不存在，自动改用 {fallback}")
    PANEL_D_METRIC = fallback

rep_order = [p["label"] for p in STATIC_REP_POINTS]

panel_d_piv = (
    point_summary_d
    .pivot_table(
        index=["label", "m_target", "E", "T"],
        columns="mode",
        values=PANEL_D_METRIC,
        aggfunc="first",
    )
    .reset_index()
)

# 只保留 static / dynamic 都有值的点
if ("static" not in panel_d_piv.columns) or ("dynamic" not in panel_d_piv.columns):
    raise RuntimeError("panel (d) 没有找到 static/dynamic 成对数据。")

panel_d_piv = panel_d_piv[panel_d_piv["label"].isin(rep_order)].copy()
panel_d_piv["label"] = pd.Categorical(panel_d_piv["label"], categories=rep_order, ordered=True)
panel_d_piv = panel_d_piv.sort_values("label").reset_index(drop=True)
panel_d_piv["label_show"] = panel_d_piv["label"].astype(str).map(_pretty_point_label)
panel_d_piv["delta_dyn_minus_static"] = panel_d_piv["dynamic"] - panel_d_piv["static"]

# 相关性和平均偏移，直接写进 panel (d)
valid_corr = panel_d_piv[["static", "dynamic"]].dropna()
if len(valid_corr) >= 2:
    rho_s = valid_corr["static"].rank().corr(valid_corr["dynamic"].rank())
    pearson_r = valid_corr["static"].corr(valid_corr["dynamic"])
else:
    rho_s = np.nan
    pearson_r = np.nan

mean_abs_shift = np.nanmean(np.abs(panel_d_piv["delta_dyn_minus_static"].values))

# 也把 panel (d) 数据单独存一下，后面写文方便
panel_d_piv.to_csv(PAPER_DIR / "Figure6_panel_d_representative_points.csv", index=False)

# =========================
# 正式画图
# =========================
fig = plt.figure(figsize=(18, 11), constrained_layout=False)
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    width_ratios=[1.25, 1.0],
    height_ratios=[1.0, 1.0],
    wspace=0.28, hspace=0.32
)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

# ---------- (a) static heatmap ----------
sns.heatmap(
    static_plot_df,
    ax=ax_a,
    annot=True,
    fmt=".2f",
    cmap="mako",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Impact = range / baseline"},
    annot_kws={"size": 10},
)
ax_a.set_title("(a) Static sensitivity heatmap", loc="left", fontsize=16, pad=10)
ax_a.set_xlabel("")
ax_a.set_ylabel("Parameter", fontsize=12)
ax_a.tick_params(axis="x", labelrotation=0, labelsize=10)
ax_a.tick_params(axis="y", labelsize=11)

# ---------- (b) R2_Psi_branches ranking ----------
sns.barplot(
    data=rank_df,
    y="param_show",
    x=rank_col,
    ax=ax_b,
    orient="h",
    color=sns.color_palette("viridis", 6)[4],
)
ax_b.set_title(r"(b) $R^2_{\Psi,\mathrm{branches}}$ ranking", loc="left", fontsize=16, pad=10)
ax_b.set_xlabel("Relative impact on R²(Psi, branches)", fontsize=12)
ax_b.set_ylabel("")
ax_b.tick_params(axis="y", labelsize=11)
ax_b.tick_params(axis="x", labelsize=10)

for container in ax_b.containers:
    ax_b.bar_label(container, fmt="%.2f", padding=4, fontsize=10)

# ---------- (c) dynamic heatmap ----------
sns.heatmap(
    dynamic_plot_df,
    ax=ax_c,
    annot=True,
    fmt=".2f",
    cmap="rocket",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Impact = range / baseline"},
    annot_kws={"size": 10},
)
ax_c.set_title("(c) Dynamic sensitivity heatmap", loc="left", fontsize=16, pad=10)
ax_c.set_xlabel("")
ax_c.set_ylabel("Parameter", fontsize=12)
ax_c.tick_params(axis="x", labelrotation=0, labelsize=10)
ax_c.tick_params(axis="y", labelsize=11)

# ---------- (d) representative-point comparison: static vs dynamic ----------
# 画 paired-point / dumbbell 风格，更适合论文
x = np.arange(len(panel_d_piv))
x_static = x - 0.10
x_dynamic = x + 0.10

# 连线
for i, row in panel_d_piv.iterrows():
    if np.isfinite(row["static"]) and np.isfinite(row["dynamic"]):
        ax_d.plot(
            [x_static[i], x_dynamic[i]],
            [row["static"], row["dynamic"]],
            lw=2.0,
            alpha=0.75,
            zorder=2,
        )

# 点
ax_d.scatter(
    x_static,
    panel_d_piv["static"],
    s=85,
    marker="o",
    label="Static m",
    zorder=3,
)
ax_d.scatter(
    x_dynamic,
    panel_d_piv["dynamic"],
    s=95,
    marker="D",
    label="Dynamic m",
    zorder=3,
)

# 在 dynamic 端标注 Δ
yr = np.nanmax([
    np.nanmax(panel_d_piv["static"].values) if len(panel_d_piv) else np.nan,
    np.nanmax(panel_d_piv["dynamic"].values) if len(panel_d_piv) else np.nan
]) - np.nanmin([
    np.nanmin(panel_d_piv["static"].values) if len(panel_d_piv) else np.nan,
    np.nanmin(panel_d_piv["dynamic"].values) if len(panel_d_piv) else np.nan
])
text_dy = 0.03 * yr if np.isfinite(yr) and yr > 0 else 0.05

for i, row in panel_d_piv.iterrows():
    if np.isfinite(row["dynamic"]) and np.isfinite(row["delta_dyn_minus_static"]):
        ax_d.text(
            x_dynamic[i] + 0.03,
            row["dynamic"] + text_dy,
            f"{row['delta_dyn_minus_static']:+.2f}",
            fontsize=9,
            ha="left",
            va="bottom",
        )

ax_d.set_xticks(x)
ax_d.set_xticklabels(panel_d_piv["label_show"], fontsize=10)
ax_d.set_ylabel(_pretty_metric_name(PANEL_D_METRIC), fontsize=12)
ax_d.set_title("(d) Representative-point comparison: static vs dynamic", loc="left", fontsize=16, pad=10)
ax_d.legend(frameon=True, fontsize=10, loc="best")

summary_text = (
    rf"$\rho_s$ = {rho_s:.2f}" if np.isfinite(rho_s) else r"$\rho_s$ = NA"
) + "\n" + (
    rf"$r$ = {pearson_r:.2f}" if np.isfinite(pearson_r) else r"$r$ = NA"
) + "\n" + (
    rf"mean |Δ| = {mean_abs_shift:.2f}" if np.isfinite(mean_abs_shift) else r"mean |Δ| = NA"
)
ax_d.text(
    0.98, 0.03, summary_text,
    transform=ax_d.transAxes,
    ha="right", va="bottom",
    fontsize=10,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9, edgecolor="0.7")
)

# ---------- 总标题 ----------
fig.suptitle(
    "Figure 6. Sensitivity and dynamic validation",
    fontsize=22,
    fontweight="bold",
    y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.965])

fig_png = PAPER_DIR / "Figure6_sensitivity_dynamic_validation.png"
fig_pdf = PAPER_DIR / "Figure6_sensitivity_dynamic_validation.pdf"

plt.savefig(fig_png, dpi=350, bbox_inches="tight")
plt.savefig(fig_pdf, bbox_inches="tight")
plt.show()

print("Saved to:")
print(fig_png)
print(fig_pdf)
print(PAPER_DIR / "Figure6_panel_d_representative_points.csv")

In [ ]:
# ============================================================
# Supplementary Figures S1–S8
# 直接基于你前面的 notebook 变量 + 原始函数来算
# 建议：加在 KMC_sensitivity_static_dynamic_stage1.ipynb 最后运行
# ============================================================

from pathlib import Path
import ast
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.optimize import curve_fit

warnings.filterwarnings("ignore")

# -----------------------------
# 0) 输出目录与开关
# -----------------------------
SUPP_DIR = OUT_DIR / "supplementary_figures_from_code"
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# True: 缺什么就直接重算
# False: 只用当前 notebook 已经存在的变量；没有就跳过
SUPP_RUN_MISSING = True

# S1 / S2 / S3 属于会重新跑 phase2 网格/单点的图，比较慢
SUPP_RUN_EXPENSIVE = True

# 你如果想把 S1 的 active-zone 深度改掉，改这里
# 这里是“z <= ?”的上限（按你截图里的 z<=11 / 15 / 19）
SUPP_ACTIVE_ZMAX_LIST = [11, 15, 19]

# S2 更多 barrier heatmaps：默认直接用你 sensitivity notebook 的 6 个代表点
SUPP_BARRIER_POINTS = STATIC_REP_POINTS

# S3 的 mT fixed-E
SUPP_MT_M_LIST = [0.00, 0.25, 0.50, 0.75, 0.90]
SUPP_MT_T_LIST = [500.0, 600.0, 700.0, 800.0, 900.0]
SUPP_MT_E_FIXED = 0.090

# baseline mE（给 S4 / S5 用）
SUPP_ME_M_LIST = [0.00, 0.25, 0.50, 0.75, 0.90]
SUPP_ME_E_LIST = [0.045, 0.060, 0.075, 0.090, 0.105, 0.120]
SUPP_ME_T_FIXED = 700.0

# quick/full 跟你原 notebook 走
SUPP_N_STRUCTURE = 2 if RUN_PRESET == "quick" else 6
SUPP_N_MC = 2 if RUN_PRESET == "quick" else 4
SUPP_N_JOBS = 1

sns.set_theme(style="whitegrid", context="talk", font_scale=1.0)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 320

generated = []
skipped = []

# ============================================================
# 1) 小工具
# ============================================================
def _ok(name, path):
    generated.append((name, str(path)))

def _skip(name, reason):
    skipped.append((name, reason))
    print(f"[skip] {name}: {reason}")

def savefig_both(fig, stem):
    png = SUPP_DIR / f"{stem}.png"
    pdf = SUPP_DIR / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    _ok(stem, png)
    plt.close(fig)

def pick_col(df, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"找不到列，候选={candidates}")
    return None

def ensure_numeric(df, cols):
    out = df.copy()
    for c in cols:
        if c in out.columns:
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def pretty_param_name(x):
    mp = {
        "SIGMA_DIS_EV": r"$\sigma_{\mathrm{dis}}$",
        "SIGMA_RES_EV": r"$\sigma_{\mathrm{res}}$",
        "P_SIGMA": r"$p_{\sigma}$",
        "LC_BETA_CELLS": r"$\beta_{lc}$",
        "LC0_CELLS": r"$lc_0$",
        "MU_X_EXTRA_EV": r"$\mu_x^{extra}$",
        "ALPHA_FIELD": r"$\alpha_E$",
        "ACTIVE_Z_MAX_FRAC": r"$z_{\mathrm{active}}^{max}$",
        "MU_SHIFT_EV": r"$\Delta \mu_z$",
        "MU_SCALE": r"$s(\mu_z)$",
        "DYN_UPDATE_EVENTS": "Dyn. update interval",
        "DYN_ALPHA_SIGMA": r"Dyn. $\alpha_{\sigma}$",
        "DYN_M_COUNT_MODE": "Dyn. m-count mode",
        "BASELINE": "Baseline",
    }
    return mp.get(str(x), str(x))

def pretty_label(lbl):
    lbl = str(lbl)
    lbl = lbl.replace("m025", "m=0.25")
    lbl = lbl.replace("m090", "m=0.90")
    lbl = lbl.replace("_sub", "\nsub")
    lbl = lbl.replace("_thr", "\nthr")
    lbl = lbl.replace("_sup", "\nsup")
    return lbl

def parse_array_cell(val):
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.array([], dtype=float)
    if isinstance(val, np.ndarray):
        return val.astype(float).ravel()
    if isinstance(val, (list, tuple)):
        try:
            return np.asarray(val, dtype=float).ravel()
        except Exception:
            return np.array([], dtype=float)

    s = str(val).strip()
    if s == "" or s.lower() == "nan":
        return np.array([], dtype=float)

    try:
        obj = ast.literal_eval(s)
        return np.asarray(obj, dtype=float).ravel()
    except Exception:
        pass

    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if len(nums) == 0:
        return np.array([], dtype=float)
    return np.asarray([float(x) for x in nums], dtype=float)

def normalized_mean_trace(arr_list, n_grid=120):
    valid = []
    for arr in arr_list:
        aa = np.asarray(arr, dtype=float).ravel()
        aa = aa[np.isfinite(aa)]
        if len(aa) >= 2:
            valid.append(aa)

    if len(valid) == 0:
        return None, None, None

    grid = np.linspace(0, 1, n_grid)
    mats = []
    for aa in valid:
        x_old = np.linspace(0, 1, len(aa))
        mats.append(np.interp(grid, x_old, aa))
    mats = np.asarray(mats, dtype=float)
    mean = np.nanmean(mats, axis=0)
    std = np.nanstd(mats, axis=0)
    return grid, mean, std

# ============================================================
# 2) scaling / E50 后处理工具（从你之前 notebook 里抽出来）
# ============================================================
KB_EV = 8.617333262145e-5

def logistic_ab(x, a, b):
    return 1.0 / (1.0 + np.exp(-(a + b * x)))

def fit_logistic_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 4:
        return None
    if (np.nanmax(y) - np.nanmin(y)) < 0.15:
        return None

    x0 = np.nanmedian(x)
    p0 = np.array([-x0 * 50.0, 50.0], dtype=float)

    try:
        popt, _ = curve_fit(logistic_ab, x, y, p0=p0, maxfev=20000)
        a, b = popt
        if abs(b) < 1e-12:
            return None
        y_pred = logistic_ab(x, a, b)
        rmse = float(np.sqrt(np.mean((y - y_pred) ** 2)))
        e50 = float(-a / b)
        wE = float(1.0 / abs(b))
        return {"a": float(a), "b": float(b), "rmse": rmse, "E50": e50, "wE": wE}
    except Exception:
        return None

def interpolate_e50(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 2:
        return np.nan

    order = np.argsort(x)
    x = x[order]
    y = y[order]
    for i in range(len(x) - 1):
        y1, y2 = y[i], y[i+1]
        if (y1 - 0.5) * (y2 - 0.5) <= 0 and abs(y2 - y1) > 1e-12:
            t = (0.5 - y1) / (y2 - y1)
            return float(x[i] + t * (x[i+1] - x[i]))
    return np.nan

def standardize_mE_summary(df_raw, fixed_T=700.0):
    out = pd.DataFrame(index=df_raw.index)
    out["source"] = "mE"
    out["m_target"] = df_raw[pick_col(df_raw, ["m_target", "m", "m_actual_mean", "m_actual"])]
    out["m_actual"] = df_raw[pick_col(df_raw, ["m_actual_mean", "m_actual", "m_target", "m"])]
    e_col = pick_col(df_raw, ["E", "E_vnm", "E_field", "Efield_vnm", "E_fixed", "E_fixed_vnm"])
    out["E"] = df_raw[e_col]
    t_col = pick_col(df_raw, ["T", "T_K", "temp_K", "temperature_K", "T_fixed", "T_fixed_K"], required=False)
    out["T"] = df_raw[t_col] if t_col is not None else fixed_T

    sigma_col = pick_col(df_raw, ["sigma_E_mean", "sigma_E", "barrier_std_z", "std_em_z"], required=False)
    out["sigma_E"] = df_raw[sigma_col] if sigma_col is not None else np.nan

    lc_col = pick_col(df_raw, ["lc_mean", "l_c", "lc", "l_c_mean"], required=False)
    out["lc"] = df_raw[lc_col] if lc_col is not None else np.nan

    delta_col = pick_col(df_raw, ["Delta_mean", "Delta"], required=False)
    if delta_col is not None:
        out["Delta"] = df_raw[delta_col]
    else:
        out["Delta"] = out["sigma_E"] / (KB_EV * out["T"])

    formed_col = pick_col(df_raw, ["formed_prob", "Pform", "formation_prob"])
    out["formed_prob"] = df_raw[formed_col]

    tset_col = pick_col(df_raw, ["log10_t_set_mean", "log10_t_set", "t_set_log10_mean"], required=False)
    out["log10_t_set"] = df_raw[tset_col] if tset_col is not None else np.nan

    out = ensure_numeric(out, ["m_target", "m_actual", "E", "T", "sigma_E", "lc", "Delta", "formed_prob", "log10_t_set"])
    return out

def standardize_mE_topology(df_raw, stage="crit", stat="median", fixed_T=700.0):
    out = pd.DataFrame(index=df_raw.index)
    out["source"] = "mE"
    out["m_target"] = df_raw[pick_col(df_raw, ["m_target", "m", "m_actual_mean", "m_actual"])]
    out["m_actual"] = df_raw[pick_col(df_raw, ["m_actual_mean", "m_actual", "m_target", "m"])]
    e_col = pick_col(df_raw, ["E", "E_vnm", "E_field", "Efield_vnm", "E_fixed", "E_fixed_vnm"])
    out["E"] = df_raw[e_col]
    t_col = pick_col(df_raw, ["T", "T_K", "temp_K", "temperature_K", "T_fixed", "T_fixed_K"], required=False)
    out["T"] = df_raw[t_col] if t_col is not None else fixed_T

    sigma_col = pick_col(df_raw, ["sigma_E_mean", "sigma_E", "barrier_std_z", "std_em_z", "sigma_real_mean"], required=False)
    out["sigma_E"] = df_raw[sigma_col] if sigma_col is not None else np.nan

    lc_col = pick_col(df_raw, ["lc_mean", "l_c", "lc", "l_c_mean"], required=False)
    out["lc"] = df_raw[lc_col] if lc_col is not None else np.nan

    delta_col = pick_col(df_raw, ["Delta_mean", "Delta"], required=False)
    if delta_col is not None:
        out["Delta"] = df_raw[delta_col]
    else:
        out["Delta"] = out["sigma_E"] / (KB_EV * out["T"])

    pform_col = pick_col(df_raw, ["formed_prob", "Pform", "formation_prob"], required=False)
    out["formed_prob"] = df_raw[pform_col] if pform_col is not None else np.nan

    stage = stage.lower()
    stat = stat.lower()

    branch_candidates = ["branches_topo_main"]
    tort_candidates = ["tortuosity_topo_main"]
    neck_candidates = ["neck_topo_main"]

    if stage == "crit":
        if stat == "median":
            branch_candidates += ["branches_crit_median_formed"]
            tort_candidates += ["tortuosity_nm_crit_median_formed"]
            neck_candidates += ["neck_nm_crit_median_formed"]
        else:
            branch_candidates += ["branches_crit_mean_formed"]
            tort_candidates += ["tortuosity_nm_crit_mean_formed"]
            neck_candidates += ["neck_nm_crit_mean_formed"]
    elif stage == "lrs":
        if stat == "median":
            branch_candidates += ["branches_lrs_median_formed", "branches_median_formed"]
            tort_candidates += ["tortuosity_nm_lrs_median_formed", "tortuosity_nm_median_formed"]
            neck_candidates += ["neck_nm_lrs_median_formed", "neck_nm_median_formed"]
        else:
            branch_candidates += ["branches_lrs_mean_formed", "branches_mean_formed"]
            tort_candidates += ["tortuosity_nm_lrs_mean_formed", "tortuosity_nm_mean_formed"]
            neck_candidates += ["neck_nm_lrs_mean_formed", "neck_nm_mean_formed"]

    out["branches"] = df_raw[pick_col(df_raw, branch_candidates)]
    out["tortuosity"] = df_raw[pick_col(df_raw, tort_candidates)]
    out["neck_nm"] = df_raw[pick_col(df_raw, neck_candidates)]

    out = ensure_numeric(out, ["m_target", "m_actual", "E", "T", "sigma_E", "lc", "Delta", "formed_prob", "branches", "tortuosity", "neck_nm"])
    return out

def add_delta_log10_tset_by_m(df):
    out = df.copy()
    out["delta_log10_t_set"] = np.nan
    for mval, g in out.groupby("m_target", dropna=False):
        sub = g.dropna(subset=["log10_t_set"]).copy()
        if len(sub) == 0:
            continue
        ref = sub["log10_t_set"].min()
        out.loc[sub.index, "delta_log10_t_set"] = sub["log10_t_set"] - ref
    return out

def fit_e50_by_m(df):
    rows = []
    fit_objs = {}
    sub_all = df.dropna(subset=["m_target", "E", "formed_prob"]).copy()

    for mval, g in sub_all.groupby("m_target", dropna=False):
        g = g.sort_values("E")
        fit = fit_logistic_1d(g["E"].values, g["formed_prob"].values)
        if fit is None:
            e50_fallback = interpolate_e50(g["E"].values, g["formed_prob"].values)
            rows.append({
                "m_target": mval,
                "n_points": len(g),
                "fit_ok": False,
                "a": np.nan,
                "b": np.nan,
                "rmse": np.nan,
                "E50": e50_fallback,
                "wE": np.nan,
            })
            fit_objs[mval] = None
        else:
            rows.append({
                "m_target": mval,
                "n_points": len(g),
                "fit_ok": True,
                "a": fit["a"],
                "b": fit["b"],
                "rmse": fit["rmse"],
                "E50": fit["E50"],
                "wE": fit["wE"],
            })
            fit_objs[mval] = fit

    e50_df = pd.DataFrame(rows).sort_values("m_target").reset_index(drop=True)
    return e50_df, fit_objs

def merge_drive_descriptors(df, e50_df):
    out = df.merge(e50_df[["m_target", "E50", "wE", "fit_ok"]], on="m_target", how="left")
    out["E_norm"] = out["E"] / out["E50"]
    out["E_hat"] = (out["E"] - out["E50"]) / out["wE"]
    return out

def fit_rmse_quadratic(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if len(x) < 4:
        return np.nan, np.nan, None, None

    X = np.column_stack([np.ones_like(x), x, x**2])
    coef, *_ = np.linalg.lstsq(X, y, rcond=None)
    y_pred = X @ coef
    rmse = float(np.sqrt(np.mean((y - y_pred) ** 2)))
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2) + 1e-12
    r2 = float(1 - ss_res / ss_tot)
    return rmse, r2, coef, y_pred

def scan_alpha_generic(df, ycol, x_col, alpha_list=None):
    if alpha_list is None:
        alpha_list = np.linspace(-1.5, 1.5, 41)

    sub = df.dropna(subset=[x_col, "Delta", "lc", ycol]).copy()
    if len(sub) < 4:
        return pd.DataFrame()

    lc_ref = float(np.nanmedian(sub["lc"].values))
    rows = []
    for alpha in alpha_list:
        desc = sub[x_col].values / (sub["Delta"].values * ((sub["lc"].values / lc_ref) ** alpha))
        rmse, r2, coef, _ = fit_rmse_quadratic(desc, sub[ycol].values)
        rows.append({
            "alpha": alpha,
            "lc_ref": lc_ref,
            "rmse": rmse,
            "r2": r2,
            "x_col": x_col,
            "y_col": ycol,
        })
    return pd.DataFrame(rows).sort_values("rmse").reset_index(drop=True)

# ============================================================
# 3) 保证 sensitivity 变量存在
# ============================================================
need_static = any(v not in globals() for v in ["static_case_metrics_df", "static_impact_df"])
need_dynamic = any(v not in globals() for v in ["dynamic_case_metrics_df", "dynamic_impact_df", "dynamic_raw_df"])

if need_static:
    if SUPP_RUN_MISSING:
        print("[supp] static sensitivity 变量不存在，开始直接重算 ...")
        static_points_df, static_case_metrics_df, static_e50_tables_df = run_static_sensitivity()
        STATIC_METRIC_COLS = [
            "formed_prob_mean6",
            "log10_t_set_med_mean6",
            "branches_crit_med_mean6",
            "tortuosity_crit_med_mean6",
            "E50_m025",
            "E50_m090",
            "R2_Psi_branches",
            "R2_Psi_tortuosity",
        ]
        static_impact_df = compute_param_impact(static_case_metrics_df, STATIC_METRIC_COLS, baseline_level_name="base")
    else:
        _skip("sensitivity_static", "缺少 static_case_metrics_df / static_impact_df，且 SUPP_RUN_MISSING=False")

if need_dynamic:
    if SUPP_RUN_MISSING:
        print("[supp] dynamic sensitivity 变量不存在，开始直接重算 ...")
        dynamic_raw_df, dynamic_case_metrics_df = run_dynamic_sensitivity()
        DYNAMIC_METRIC_COLS = [
            "delta_formed_prob_mean6",
            "delta_log10_t_set_median_formed_mean6",
            "delta_branches_crit_median_formed_mean6",
            "delta_tortuosity_nm_crit_median_formed_mean6",
            "delta_m_final_mean_mean6",
            "delta_Delta_final_mean_mean6",
            "abs_delta_formed_prob_mean6",
            "abs_delta_log10_t_set_median_formed_mean6",
            "abs_delta_branches_crit_median_formed_mean6",
        ]
        dynamic_impact_df = compute_param_impact(dynamic_case_metrics_df, DYNAMIC_METRIC_COLS, baseline_level_name="base")
    else:
        _skip("sensitivity_dynamic", "缺少 dynamic_case_metrics_df / dynamic_impact_df / dynamic_raw_df，且 SUPP_RUN_MISSING=False")

# ============================================================
# 4) baseline phase2 namespace 与 baseline mE / mT 数据
# ============================================================
def load_phase2_ns(extra_overrides=None, save_subdir="_supp_phase2_tmp"):
    overrides = dict(BASELINE_STATIC)
    overrides["SAVE_DIR"] = SUPP_DIR / save_subdir
    overrides["FAST_MODE"] = (RUN_PRESET == "quick")
    if extra_overrides is not None:
        overrides.update(extra_overrides)
    return load_notebook_namespace(STATIC_NOTEBOOK, overrides)

phase2_base_ns = None
df_mE_base = None
summary_mE_base = None
df_me_sum_std = None
df_me_topo_std = None
df_me_sum_dn = None
df_me_topo_dn = None
e50_df = None
fit_objs = None
score_psi_br = None
score_psi_to = None
score_xi_br = None
score_xi_to = None

if SUPP_RUN_EXPENSIVE:
    print("[supp] loading baseline phase2 namespace ...")
    phase2_base_ns = load_phase2_ns(save_subdir="_supp_phase2_base")

    print("[supp] running baseline mE grid for S4/S5 ...")
    df_mE_base = phase2_base_ns["run_phase2_grid"](
        mode="mE",
        m_list=SUPP_ME_M_LIST,
        E_list=SUPP_ME_E_LIST,
        T_fixed=SUPP_ME_T_FIXED,
        n_structure_seeds=SUPP_N_STRUCTURE,
        n_mc_per_structure=SUPP_N_MC,
        n_jobs=SUPP_N_JOBS,
    )
    summary_mE_base = phase2_base_ns["summarize_phase2"](df_mE_base, mode="mE")

    df_me_sum_std = standardize_mE_summary(summary_mE_base, fixed_T=SUPP_ME_T_FIXED)
    df_me_sum_std = add_delta_log10_tset_by_m(df_me_sum_std)

    df_me_topo_std = standardize_mE_topology(summary_mE_base, stage="crit", stat="median", fixed_T=SUPP_ME_T_FIXED)

    e50_df, fit_objs = fit_e50_by_m(df_me_sum_std)
    df_me_sum_dn = merge_drive_descriptors(df_me_sum_std, e50_df)
    df_me_topo_dn = merge_drive_descriptors(df_me_topo_std, e50_df)

    score_psi_br = scan_alpha_generic(df_me_topo_dn, ycol="branches", x_col="E_norm")
    score_psi_to = scan_alpha_generic(df_me_topo_dn, ycol="tortuosity", x_col="E_norm")

    # 这里 Xi 用的是基于 E_hat 的 threshold-centered descriptor
    # 这是按你现有 notebook 里已经有的 E_hat 接着定义的
    score_xi_br = scan_alpha_generic(df_me_topo_dn, ycol="branches", x_col="E_hat")
    score_xi_to = scan_alpha_generic(df_me_topo_dn, ycol="tortuosity", x_col="E_hat")

# ============================================================
# 5) Figure S1 — active-zone depth comparison
# ============================================================
def make_S1():
    if not SUPP_RUN_EXPENSIVE:
        _skip("Figure_S1", "SUPP_RUN_EXPENSIVE=False")
        return

    rows = []
    topo_rows = []

    base_tmp = load_phase2_ns(save_subdir="_supp_phase2_probe")
    z_ox_end = int(base_tmp["Z_OX_END"])

    for zmax in SUPP_ACTIVE_ZMAX_LIST:
        frac = float(zmax) / float(z_ox_end)
        ns = load_phase2_ns(
            extra_overrides={"ACTIVE_Z_MAX_FRAC": frac},
            save_subdir=f"_supp_active_zone_z{zmax}"
        )
        print(f"[supp:S1] running active-zone z<={zmax} (frac={frac:.4f})")

        df = ns["run_phase2_grid"](
            mode="mE",
            m_list=[0.25, 0.90],
            E_list=SUPP_ME_E_LIST,
            T_fixed=SUPP_ME_T_FIXED,
            n_structure_seeds=SUPP_N_STRUCTURE,
            n_mc_per_structure=SUPP_N_MC,
            n_jobs=SUPP_N_JOBS,
        )
        summ = ns["summarize_phase2"](df, mode="mE")
        summ["zone_tag"] = f"z<={zmax}"
        rows.append(summ)

        topo = standardize_mE_topology(summ, stage="crit", stat="median", fixed_T=SUPP_ME_T_FIXED)
        topo["zone_tag"] = f"z<={zmax}"
        topo_rows.append(topo)

    all_df = pd.concat(rows, ignore_index=True)
    all_topo = pd.concat(topo_rows, ignore_index=True)

    std_sum = standardize_mE_summary(all_df, fixed_T=SUPP_ME_T_FIXED)

    e50_rows = []
    for zone_tag, g0 in std_sum.groupby("zone_tag"):
        e50_tmp, _ = fit_e50_by_m(g0)
        e50_tmp["zone_tag"] = zone_tag
        e50_rows.append(e50_tmp)
    e50_all = pd.concat(e50_rows, ignore_index=True)

    fig, axes = plt.subplots(2, 2, figsize=(13.8, 9.6))
    axes = axes.ravel()

    # (a) m=0.25
    ax = axes[0]
    sub = std_sum[np.isclose(std_sum["m_target"], 0.25)]
    for tag, g in sub.groupby("zone_tag"):
        g = g.sort_values("E")
        ax.plot(g["E"], g["formed_prob"], marker="o", linewidth=2, label=tag)
    ax.axhline(0.5, ls="--", lw=1.1, color="gray")
    ax.set_title("(a) Formation curves at m = 0.25", loc="left")
    ax.set_xlabel("E (V/nm)")
    ax.set_ylabel("Formation probability")
    ax.legend()

    # (b) m=0.90
    ax = axes[1]
    sub = std_sum[np.isclose(std_sum["m_target"], 0.90)]
    for tag, g in sub.groupby("zone_tag"):
        g = g.sort_values("E")
        ax.plot(g["E"], g["formed_prob"], marker="o", linewidth=2, label=tag)
    ax.axhline(0.5, ls="--", lw=1.1, color="gray")
    ax.set_title("(b) Formation curves at m = 0.90", loc="left")
    ax.set_xlabel("E (V/nm)")
    ax.set_ylabel("Formation probability")
    ax.legend()

    # (c) E50 comparison
    ax = axes[2]
    sns.pointplot(data=e50_all, x="zone_tag", y="E50", hue="m_target", dodge=0.30, markers=["o", "s"], ax=ax)
    ax.set_title("(c) E50 vs active-zone definition", loc="left")
    ax.set_xlabel("Active-zone definition")
    ax.set_ylabel("E50 (V/nm)")
    ax.legend(title="m")

    # (d) branches near E50
    ax = axes[3]
    near_rows = []
    for tag, g in all_topo.groupby("zone_tag"):
        for m0 in [0.25, 0.90]:
            gg = g[np.isclose(g["m_target"], m0)].copy()
            e50_sub = e50_all[(e50_all["zone_tag"] == tag) & (np.isclose(e50_all["m_target"], m0))]
            if len(gg) == 0 or len(e50_sub) == 0 or not np.isfinite(float(e50_sub.iloc[0]["E50"])):
                continue
            e50v = float(e50_sub.iloc[0]["E50"])
            gg["dist"] = np.abs(gg["E"] - e50v)
            row = gg.sort_values("dist").iloc[0]
            near_rows.append({
                "zone_tag": tag,
                "m_target": m0,
                "branches": row["branches"],
                "tortuosity": row["tortuosity"],
            })
    near_df = pd.DataFrame(near_rows)
    if len(near_df):
        sns.pointplot(data=near_df, x="zone_tag", y="branches", hue="m_target", dodge=0.30, markers=["o", "s"], ax=ax)
        ax.set_title("(d) Branches near E ≈ E50", loc="left")
        ax.set_xlabel("Active-zone definition")
        ax.set_ylabel("Branches")
        ax.legend(title="m")
    else:
        ax.text(0.5, 0.5, "No valid rows near E50", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()

    fig.suptitle("Figure S1. Active-zone depth definition comparison", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    savefig_both(fig, "Figure_S1_active_zone_depth_comparison")

# ============================================================
# 6) Figure S2 — more barrier heatmaps
# ============================================================
def make_S2():
    if not SUPP_RUN_EXPENSIVE:
        _skip("Figure_S2", "SUPP_RUN_EXPENSIVE=False")
        return

    if phase2_base_ns is None:
        _skip("Figure_S2", "phase2_base_ns 不存在")
        return

    outs = []
    for i, pt in enumerate(SUPP_BARRIER_POINTS):
        out = phase2_base_ns["simulate_one_phase2"](
            m0_target=float(pt["m0"]),
            E_vnm=float(pt["E"]),
            T=float(pt["T"]),
            structure_seed=2025 + 100 * i,
            mc_seed=50000 + i,
            return_snapshots=False,
            store_rate_samples=False,
        )
        outs.append({
            "label": pt["label"],
            "m0": float(pt["m0"]),
            "E": float(pt["E"]),
            "E_over_E50_target": pt.get("E_over_E50_target", np.nan),
            "em_z": out["em_z"],
            "em_x": out["em_x"],
            "eta": out["eta"],
        })

    vmin = min(np.nanmin(x["em_z"]) for x in outs)
    vmax = max(np.nanmax(x["em_z"]) for x in outs)

    fig, axes = plt.subplots(2, 3, figsize=(15.0, 8.4))
    axes = axes.ravel()

    for ax, row in zip(axes, outs):
        im = ax.imshow(row["em_z"].T, origin="lower", aspect="auto", vmin=vmin, vmax=vmax)
        ax.set_title(f"{pretty_label(row['label'])}\nm={row['m0']:.2f}, E={row['E']:.3f}, E/E50≈{row['E_over_E50_target']:.2f}")
        ax.set_xlabel("x cell")
        ax.set_ylabel("z cell")

    cbar = fig.colorbar(im, ax=axes.tolist(), fraction=0.025, pad=0.02)
    cbar.set_label("Vertical migration barrier $E_m^z$ (eV)")

    fig.suptitle("Figure S2. Additional barrier heatmaps", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    savefig_both(fig, "Figure_S2_more_barrier_heatmaps")

# ============================================================
# 7) Figure S3 — more mT fixed-E results
# ============================================================
def make_S3():
    if not SUPP_RUN_EXPENSIVE:
        _skip("Figure_S3", "SUPP_RUN_EXPENSIVE=False")
        return

    ns = load_phase2_ns(save_subdir="_supp_mT")
    print("[supp:S3] running mT fixed-E grid ...")
    df_mT = ns["run_phase2_grid"](
        mode="mT",
        m_list=SUPP_MT_M_LIST,
        T_list=SUPP_MT_T_LIST,
        E_fixed=SUPP_MT_E_FIXED,
        n_structure_seeds=SUPP_N_STRUCTURE,
        n_mc_per_structure=SUPP_N_MC,
        n_jobs=SUPP_N_JOBS,
    )
    summary_mT = ns["summarize_phase2"](df_mT, mode="mT")

    t_col = "T"
    panels = [
        ("formed_prob", "Formation probability"),
        ("log10_t_set_mean", "mean log10(tset / s)"),
        ("branches_crit_median_formed", "Critical branches"),
        ("tortuosity_nm_crit_median_formed", "Critical tortuosity"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(13.8, 9.6))
    axes = axes.ravel()
    pal = sns.color_palette("viridis", n_colors=len(sorted(summary_mT["m_target"].dropna().unique())))

    for ax, (col, ttl) in zip(axes, panels):
        if col not in summary_mT.columns:
            ax.text(0.5, 0.5, f"Missing:\n{col}", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        for c, (m0, g) in zip(pal, sorted(summary_mT.groupby("m_target"), key=lambda x: x[0])):
            g = g.sort_values(t_col)
            ax.plot(g[t_col], g[col], marker="o", linewidth=2, color=c, label=f"m={m0:.2f}")
        ax.set_title(ttl)
        ax.set_xlabel("T (K)")
        ax.set_ylabel("")

    axes[0].legend(ncol=2, fontsize=10)

    fig.suptitle("Figure S3. Additional mT fixed-E results", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    savefig_both(fig, "Figure_S3_more_mT_fixedE_results")

# ============================================================
# 8) Figure S4 — tortuosity scaling suite
# ============================================================
def make_S4():
    if df_me_topo_dn is None or score_psi_to is None or len(score_psi_to) == 0:
        _skip("Figure_S4", "缺少 baseline mE scaling 数据")
        return

    fig, axes = plt.subplots(2, 2, figsize=(13.8, 10.0))
    axes = axes.ravel()

    # (a)
    ax = axes[0]
    sub = df_me_topo_dn.dropna(subset=["E_norm", "tortuosity"]).copy()
    sns.scatterplot(data=sub, x="E_norm", y="tortuosity", hue="Delta", size="lc", palette="magma", sizes=(60, 220), ax=ax)
    ax.set_title("(a) Tortuosity vs normalized drive", loc="left")
    ax.set_xlabel(r"$E/E_{50}$")
    ax.set_ylabel("Tortuosity")

    # (b)
    ax = axes[1]
    sub = df_me_topo_dn.dropna(subset=["Delta", "E_norm", "tortuosity"]).copy()
    sns.scatterplot(data=sub, x="Delta", y="E_norm", hue="tortuosity", size="lc", palette="plasma", sizes=(60, 220), ax=ax)
    ax.set_title("(b) Tortuosity in (Δ, E/E50) space", loc="left")
    ax.set_xlabel(r"$\Delta = \sigma_E/(k_B T)$")
    ax.set_ylabel(r"$E/E_{50}$")

    # (c)
    ax = axes[2]
    tmp = score_psi_to.sort_values("alpha")
    ax.plot(tmp["alpha"], tmp["r2"], marker="o", linewidth=2)
    if len(tmp):
        best = tmp.sort_values("r2", ascending=False).iloc[0]
        ax.axvline(best["alpha"], ls="--", lw=1.1, color="gray")
        ax.text(best["alpha"], best["r2"], f"best α={best['alpha']:.2f}", fontsize=10, ha="left", va="bottom")
    ax.set_title("(c) Alpha scan for tortuosity", loc="left")
    ax.set_xlabel(r"$\alpha$")
    ax.set_ylabel(r"$R^2$")

    # (d)
    ax = axes[3]
    best = score_psi_to.sort_values("r2", ascending=False).iloc[0]
    alpha_best = float(best["alpha"])
    sub = df_me_topo_dn.dropna(subset=["E_norm", "Delta", "lc", "tortuosity"]).copy()
    lc_ref = float(np.nanmedian(sub["lc"].values))
    sub["Psi_best"] = sub["E_norm"] / (sub["Delta"] * ((sub["lc"] / lc_ref) ** alpha_best))
    rmse, r2, coef, _ = fit_rmse_quadratic(sub["Psi_best"].values, sub["tortuosity"].values)

    sns.scatterplot(data=sub, x="Psi_best", y="tortuosity", hue="m_target", palette="viridis", s=85, ax=ax)
    if coef is not None:
        xx = np.linspace(np.nanmin(sub["Psi_best"]), np.nanmax(sub["Psi_best"]), 300)
        yy = coef[0] + coef[1] * xx + coef[2] * xx**2
        ax.plot(xx, yy, color="black", lw=2, label=f"quad fit, R²={r2:.3f}")
        ax.legend()
    ax.set_title("(d) Best Psi collapse for tortuosity", loc="left")
    ax.set_xlabel(r"$\Psi_\alpha = (E/E_{50})/[\Delta(l_c/l_{c,ref})^\alpha]$")
    ax.set_ylabel("Tortuosity")

    fig.suptitle("Figure S4. Tortuosity scaling suite", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    savefig_both(fig, "Figure_S4_tortuosity_scaling_suite")

# ============================================================
# 9) Figure S5 — alpha scan & Psi/Xi comparison
# Xi 这里按你已有 notebook 里的 E_hat 接成 threshold-centered descriptor
# ============================================================
def make_S5():
    if any(x is None for x in [score_psi_br, score_psi_to, score_xi_br, score_xi_to]):
        _skip("Figure_S5", "缺少 baseline scaling 扫描结果")
        return

    fig, axes = plt.subplots(1, 2, figsize=(13.8, 5.4))
    ax1, ax2 = axes

    # (a) branches
    if len(score_psi_br):
        t = score_psi_br.sort_values("alpha")
        ax1.plot(t["alpha"], t["r2"], marker="o", linewidth=2, label="Psi → branches")
    if len(score_xi_br):
        t = score_xi_br.sort_values("alpha")
        ax1.plot(t["alpha"], t["r2"], marker="s", linewidth=2, label="Xi → branches")
    ax1.set_title("(a) Alpha scan for branch descriptor", loc="left")
    ax1.set_xlabel(r"$\alpha$")
    ax1.set_ylabel(r"$R^2$")
    ax1.legend()

    # (b) tortuosity
    if len(score_psi_to):
        t = score_psi_to.sort_values("alpha")
        ax2.plot(t["alpha"], t["r2"], marker="o", linewidth=2, label="Psi → tortuosity")
    if len(score_xi_to):
        t = score_xi_to.sort_values("alpha")
        ax2.plot(t["alpha"], t["r2"], marker="s", linewidth=2, label="Xi → tortuosity")
    ax2.set_title("(b) Alpha scan for tortuosity descriptor", loc="left")
    ax2.set_xlabel(r"$\alpha$")
    ax2.set_ylabel(r"$R^2$")
    ax2.legend()

    fig.suptitle("Figure S5. Alpha scan and Psi/Xi comparison", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    savefig_both(fig, "Figure_S5_alpha_scan_Psi_Xi_comparison")

# ============================================================
# 10) Figure S6 — static OAT raw trajectories
# ============================================================
def make_S6():
    if "static_case_metrics_df" not in globals():
        _skip("Figure_S6", "缺少 static_case_metrics_df")
        return

    metrics = [
        ("formed_prob_mean6", "Formation probability"),
        ("branches_crit_med_mean6", "Critical branches"),
        ("E50_m025", r"$E_{50}$ at m=0.25"),
        ("E50_m090", r"$E_{50}$ at m=0.90"),
    ]

    fig, axes = plt.subplots(2, 2, figsize=(14.0, 9.8))
    axes = axes.ravel()

    for ax, (col, ttl) in zip(axes, metrics):
        if col not in static_case_metrics_df.columns:
            ax.text(0.5, 0.5, f"Missing:\n{col}", ha="center", va="center", transform=ax.transAxes)
            ax.set_axis_off()
            continue
        sns.lineplot(data=static_case_metrics_df, x="level_name", y=col, hue="param", marker="o", linewidth=2, ax=ax)
        ax.set_title(ttl)
        ax.set_xlabel("Level")
        ax.set_ylabel("")
    axes[0].legend(title="Parameter", fontsize=9)
    for ax in axes[1:]:
        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

    fig.suptitle("Figure S6. Static OAT raw trajectories", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    savefig_both(fig, "Figure_S6_static_OAT_raw_trajectories")

# ============================================================
# 11) Figure S7 — dynamic parameter sensitivity
# ============================================================
def make_S7():
    if "dynamic_impact_df" not in globals():
        _skip("Figure_S7", "缺少 dynamic_impact_df")
        return

    metric_cols = [
        "abs_delta_formed_prob_mean6",
        "abs_delta_log10_t_set_median_formed_mean6",
        "abs_delta_branches_crit_median_formed_mean6",
        "delta_m_final_mean_mean6",
        "delta_Delta_final_mean_mean6",
    ]
    use_cols = [c for c in metric_cols if c in dynamic_impact_df.columns]
    if len(use_cols) == 0:
        _skip("Figure_S7", "dynamic_impact_df 里没有目标列")
        return

    plot_df = dynamic_impact_df.copy()
    plot_df["param_show"] = plot_df["param"].map(pretty_param_name)

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.7))
    ax1, ax2 = axes

    hm = plot_df.set_index("param_show")[use_cols].rename(columns={
        "abs_delta_formed_prob_mean6": "|Δ formed prob.|",
        "abs_delta_log10_t_set_median_formed_mean6": "|Δ log10(tset)|",
        "abs_delta_branches_crit_median_formed_mean6": "|Δ branches@crit|",
        "delta_m_final_mean_mean6": "Δ m_final",
        "delta_Delta_final_mean_mean6": "Δ Delta_final",
    })
    sns.heatmap(hm, annot=True, fmt=".2f", cmap="rocket", linewidths=0.8, linecolor="white", ax=ax1)
    ax1.set_title("(a) Dynamic sensitivity heatmap", loc="left")
    ax1.set_xlabel("")
    ax1.set_ylabel("")

    rank_col = "abs_delta_branches_crit_median_formed_mean6"
    if rank_col in plot_df.columns:
        tmp = plot_df[["param_show", rank_col]].dropna().sort_values(rank_col, ascending=False)
        sns.barplot(data=tmp, y="param_show", x=rank_col, ax=ax2)
        ax2.set_title("(b) Ranking by |Δ branches@crit|", loc="left")
        ax2.set_xlabel("|Δ branches@crit|")
        ax2.set_ylabel("")
    else:
        ax2.text(0.5, 0.5, f"Missing:\n{rank_col}", ha="center", va="center", transform=ax2.transAxes)
        ax2.set_axis_off()

    fig.suptitle("Figure S7. Dynamic parameter sensitivity", fontsize=20, fontweight="bold", y=1.01)
    plt.tight_layout()
    savefig_both(fig, "Figure_S7_dynamic_parameter_sensitivity")

# ============================================================
# 12) Figure S8 — representative-point static/dynamic traces
# ============================================================
def make_S8():
    if "dynamic_raw_df" not in globals():
        _skip("Figure_S8", "缺少 dynamic_raw_df")
        return

    df = dynamic_raw_df.copy()

    # 只取 baseline / base
    if "param" in df.columns:
        df = df[df["param"] == "BASELINE"].copy()
    if "level_name" in df.columns:
        df = df[df["level_name"] == "base"].copy()

    if len(df) == 0:
        _skip("Figure_S8", "baseline dynamic_raw_df 为空")
        return

    for c in ["trace_m", "trace_sigma", "trace_step", "trace_time"]:
        if c in df.columns:
            df[c] = df[c].apply(parse_array_cell)

    if "label" not in df.columns or "mode" not in df.columns:
        _skip("Figure_S8", "dynamic_raw_df 缺少 label / mode")
        return

    labels = list(pd.unique(df["label"]))
    labels = [x for x in STATIC_REP_POINTS if x["label"] in labels]
    labels = [x["label"] for x in labels]

    ncols = 2
    nrows = int(np.ceil(len(labels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(14.5, 4.3 * nrows), squeeze=False)
    axes = axes.ravel()

    for ax, lbl in zip(axes, labels):
        sub = df[df["label"] == lbl].copy()
        dyn = sub[sub["mode"].astype(str).str.lower() == "dynamic"].copy()
        sta = sub[sub["mode"].astype(str).str.lower() == "static"].copy()

        m_grid, m_mean, m_std = (None, None, None)
        s_grid, s_mean, s_std = (None, None, None)

        if "trace_m" in dyn.columns:
            m_grid, m_mean, m_std = normalized_mean_trace(dyn["trace_m"].tolist(), n_grid=120)
        if "trace_sigma" in dyn.columns:
            s_grid, s_mean, s_std = normalized_mean_trace(dyn["trace_sigma"].tolist(), n_grid=120)

        static_m = np.nan
        if "m_actual" in sta.columns and len(sta):
            static_m = float(np.nanmean(sta["m_actual"]))
        elif "m_target" in sta.columns and len(sta):
            static_m = float(np.nanmean(sta["m_target"]))

        static_sigma = np.nan
        for col in ["sigma_E", "sigma_final", "Delta"]:
            if col in sta.columns and len(sta):
                static_sigma = float(np.nanmean(sta[col]))
                break

        # 左轴: m(t)
        if m_grid is not None:
            ax.plot(m_grid, m_mean, linewidth=2.3, label="dynamic m(t)")
            ax.fill_between(m_grid, m_mean - m_std, m_mean + m_std, alpha=0.20)
        if np.isfinite(static_m):
            ax.axhline(static_m, ls="--", lw=1.6, label="static m")

        ax.set_title(pretty_label(lbl))
        ax.set_xlabel("Normalized trace progress")
        ax.set_ylabel("m(t)")

        # 右轴: sigma(t)
        ax2 = ax.twinx()
        if s_grid is not None:
            ax2.plot(s_grid, s_mean, linewidth=2.0, linestyle="-", label=r"dynamic $\sigma_E(t)$")
            ax2.fill_between(s_grid, s_mean - s_std, s_mean + s_std, alpha=0.15)
        if np.isfinite(static_sigma):
            ax2.axhline(static_sigma, ls=":", lw=1.6, label=r"static ref")

        ax2.set_ylabel(r"$\sigma_E(t)$ / ref")

        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        if len(h1) + len(h2) > 0:
            ax.legend(h1 + h2, l1 + l2, loc="best", fontsize=9)

    for ax in axes[len(labels):]:
        ax.set_axis_off()

    fig.suptitle("Figure S8. Representative-point static/dynamic traces", fontsize=20, fontweight="bold", y=0.995)
    plt.tight_layout(rect=[0, 0, 1, 0.98])
    savefig_both(fig, "Figure_S8_representative_point_static_dynamic_traces")

# ============================================================
# 13) 开始出图
# ============================================================
print("BASE_DIR        =", BASE_DIR)
print("STATIC_NOTEBOOK =", STATIC_NOTEBOOK)
print("DYNAMIC_NOTEBOOK=", DYNAMIC_NOTEBOOK)
print("SUPP_DIR        =", SUPP_DIR)
print()

make_S1()
make_S2()
make_S3()
make_S4()
make_S5()
make_S6()
make_S7()
make_S8()

print("\n================ Generated ================\n")
if len(generated) == 0:
    print("No figure was generated.")
else:
    for name, path in generated:
        print(f"[ok] {name}: {path}")

print("\n================ Skipped ================\n")
if len(skipped) == 0:
    print("None")
else:
    for name, reason in skipped:
        print(f"[skip] {name}: {reason}")

## PATCH cells added by ChatGPT
运行下面两个新代码单元即可覆盖 Figure 6 和 S1 的旧版本。

In [ ]:

# ============================================================
# PATCH — Figure 6 (recommended paper version)
# 运行方式：直接在原 notebook 最后新建一个 cell，粘贴并运行
# ============================================================

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

required_vars = [
    "OUT_DIR",
    "static_impact_df",
    "dynamic_impact_df",
    "dynamic_raw_df",
    "summarize_runs",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"缺少这些变量，前面的单元还没跑完：{missing}")

PAPER_DIR = OUT_DIR / "paper_figures_revised"
PAPER_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.92)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 350

# ----------------------------
# 可调参数
# ----------------------------
# 去掉 E50(m=0.90)，因为它会把一个异常高值（21.05）带进热图
STATIC_PANEL_A_COLS = [
    "formed_prob_mean6",
    "log10_t_set_med_mean6",
    "branches_crit_med_mean6",
    "tortuosity_crit_med_mean6",
    "E50_m025",
    "R2_Psi_branches",
]

DYNAMIC_PANEL_C_COLS = [
    "abs_delta_formed_prob_mean6",
    "abs_delta_log10_t_set_median_formed_mean6",
    "abs_delta_branches_crit_median_formed_mean6",
    "delta_m_final_mean_mean6",
    "delta_Delta_final_mean_mean6",
]

# panel (d) 改成 formed_prob，更稳，不会像 branches@crit 那样大量掉点
PANEL_D_METRIC = "formed_prob"

# 默认把 m=0.90 排除出 panel (d)
PANEL_D_EXCLUDE_M = {0.90}

metric_label_map = {
    "formed_prob_mean6": "Formation\nprobability",
    "log10_t_set_med_mean6": r"$\log_{10}(t_{\mathrm{set}})$",
    "branches_crit_med_mean6": "Branches at\ncritical state",
    "tortuosity_crit_med_mean6": "Tortuosity at\ncritical state",
    "E50_m025": r"$E_{50}$ at $m=0.25$",
    "R2_Psi_branches": r"$R^2(\Psi,\mathrm{branches})$",
    "abs_delta_formed_prob_mean6": r"$|\Delta P_{\mathrm{form}}|$",
    "abs_delta_log10_t_set_median_formed_mean6": r"$|\Delta \log_{10}(t_{\mathrm{set}})|$",
    "abs_delta_branches_crit_median_formed_mean6": r"$|\Delta \mathrm{Branches}_{\mathrm{crit}}|$",
    "delta_m_final_mean_mean6": r"$\Delta m_{\mathrm{final}}$",
    "delta_Delta_final_mean_mean6": r"$\Delta \Delta_{\mathrm{final}}$",
    "branches_crit_median_formed": "Branches at critical state",
    "tortuosity_nm_crit_median_formed": "Tortuosity at critical state (nm)",
    "m_final_mean": r"Final $m$",
    "formed_prob": "Formation probability",
    "log10_t_set_median_formed": r"$\log_{10}(t_{\mathrm{set}})$ (formed)",
}

param_label_map = {
    "SIGMA_DIS_EV": r"$\sigma_{\mathrm{dis}}$",
    "SIGMA_RES_EV": r"$\sigma_{\mathrm{res}}$",
    "P_SIGMA": r"$p_0$",
    "LC_BETA_CELLS": r"$\beta_{\mathrm{lc}}$",
    "LC0_CELLS": r"$lc_0$",
    "MU_X_EXTRA_EV": r"$\mu_x^{\mathrm{extra}}$",
    "ALPHA_FIELD": r"$\alpha_E$",
    "ACTIVE_Z_MAX_FRAC": r"$z_{\mathrm{active}}^{\max}$",
    "MU_SHIFT_EV": r"$\Delta \mu_z$",
    "MU_SCALE": r"$s(\mu_z)$",
    "DYN_UPDATE_EVENTS": "Dyn. update\ninterval",
    "DYN_ALPHA_SIGMA": r"Dyn. $\alpha_{\sigma}$",
    "DYN_M_COUNT_MODE": "Dyn. m-count\nmode",
    "BASELINE": "Baseline",
}

def _pretty_param_name(x):
    return param_label_map.get(x, str(x).replace("_", "\n"))

def _pretty_metric_name(x):
    return metric_label_map.get(x, x)

def _pretty_point_label(lbl):
    lbl = str(lbl)
    lbl = lbl.replace("m025", "m=0.25")
    lbl = lbl.replace("m050", "m=0.50")
    lbl = lbl.replace("m075", "m=0.75")
    lbl = lbl.replace("m090", "m=0.90")
    lbl = lbl.replace("_sub", "\nsub")
    lbl = lbl.replace("_thr", "\nthr")
    lbl = lbl.replace("_sup", "\nsup")
    return lbl

def _safe_save(fig, png_path, pdf_path):
    png_path = Path(png_path)
    pdf_path = Path(pdf_path)
    png_path.parent.mkdir(parents=True, exist_ok=True)
    pdf_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        fig.savefig(png_path, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf_path, bbox_inches="tight", facecolor="white")
    except Exception as e:
        fallback_dir = OUT_DIR / "fig_tmp"
        fallback_dir.mkdir(parents=True, exist_ok=True)
        png_f = fallback_dir / png_path.name
        pdf_f = fallback_dir / pdf_path.name
        print(f"[save fallback] {e}")
        fig.savefig(png_f, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf_f, bbox_inches="tight", facecolor="white")
        return png_f, pdf_f
    return png_path, pdf_path

# ----------------------------
# panel (a): static heatmap
# ----------------------------
static_plot_df = static_impact_df.copy()
static_plot_df = static_plot_df[static_plot_df["param"] != "BASELINE"].copy()
static_cols_use = [c for c in STATIC_PANEL_A_COLS if c in static_plot_df.columns]

static_plot_df = (
    static_plot_df.set_index("param")[static_cols_use]
    .rename(index=_pretty_param_name)
    .rename(columns=_pretty_metric_name)
)

# ----------------------------
# panel (b): ranking
# ----------------------------
rank_col = "R2_Psi_branches"
rank_df = static_impact_df.copy()
rank_df = rank_df[(rank_df["param"] != "BASELINE") & rank_df[rank_col].notna()].copy()
rank_df["param_show"] = rank_df["param"].map(_pretty_param_name)
rank_df = rank_df.sort_values(rank_col, ascending=False)

# ----------------------------
# panel (c): dynamic heatmap
# ----------------------------
dynamic_plot_df = dynamic_impact_df.copy()
dynamic_plot_df = dynamic_plot_df[dynamic_plot_df["param"] != "BASELINE"].copy()
dynamic_cols_use = [c for c in DYNAMIC_PANEL_C_COLS if c in dynamic_plot_df.columns]
dynamic_plot_df = (
    dynamic_plot_df.set_index("param")[dynamic_cols_use]
    .rename(index=_pretty_param_name)
    .rename(columns=_pretty_metric_name)
)

# ----------------------------
# panel (d): static vs dynamic robustness
# ----------------------------
base_dyn = dynamic_raw_df.copy()
if "param" in base_dyn.columns:
    base_dyn = base_dyn[base_dyn["param"] == "BASELINE"].copy()
if "level_name" in base_dyn.columns:
    base_dyn = base_dyn[base_dyn["level_name"] == "base"].copy()

point_summary_d = summarize_runs(
    base_dyn,
    group_cols=("label", "mode", "m_target", "E", "T")
).copy()

fallback_candidates = [
    PANEL_D_METRIC,
    "formed_prob",
    "m_final_mean",
    "log10_t_set_median_formed",
    "branches_crit_median_formed",
]
fallback = next((c for c in fallback_candidates if c in point_summary_d.columns), None)
if fallback is None:
    raise RuntimeError("panel (d) 没有找到可用指标。")
PANEL_D_METRIC = fallback

panel_d_piv = (
    point_summary_d
    .pivot_table(
        index=["label", "m_target", "E", "T"],
        columns="mode",
        values=PANEL_D_METRIC,
        aggfunc="first",
    )
    .reset_index()
)

if ("static" not in panel_d_piv.columns) or ("dynamic" not in panel_d_piv.columns):
    raise RuntimeError("panel (d) 没有找到 static/dynamic 成对数据。")

# 先剔除 m=0.90
panel_d_piv0 = panel_d_piv[~panel_d_piv["m_target"].isin(PANEL_D_EXCLUDE_M)].copy()
valid0 = panel_d_piv0[["static", "dynamic"]].dropna()

# 如果剔除后点太少，就回退到不过滤
if len(valid0) >= 2:
    panel_d_piv = panel_d_piv0.copy()

panel_d_piv = panel_d_piv.dropna(subset=["static", "dynamic"]).copy()
if len(panel_d_piv) == 0:
    raise RuntimeError("panel (d) 过滤后没有有效的 static/dynamic 成对点。")

panel_d_piv["label_show"] = panel_d_piv["label"].astype(str).map(_pretty_point_label)
panel_d_piv["delta_dyn_minus_static"] = panel_d_piv["dynamic"] - panel_d_piv["static"]
panel_d_piv = panel_d_piv.sort_values(["m_target", "E", "label_show"]).reset_index(drop=True)

valid_corr = panel_d_piv[["static", "dynamic"]].dropna()
if len(valid_corr) >= 2:
    rho_s = valid_corr["static"].rank().corr(valid_corr["dynamic"].rank())
    pearson_r = valid_corr["static"].corr(valid_corr["dynamic"])
else:
    rho_s = np.nan
    pearson_r = np.nan
mean_abs_shift = np.nanmean(np.abs(panel_d_piv["delta_dyn_minus_static"].values))

panel_d_piv.to_csv(PAPER_DIR / "Figure6_panel_d_points_revised.csv", index=False)

# ----------------------------
# plot
# ----------------------------
fig = plt.figure(figsize=(17, 10), constrained_layout=False)
gs = fig.add_gridspec(
    nrows=2, ncols=2,
    width_ratios=[1.15, 1.0],
    height_ratios=[1.0, 1.0],
    wspace=0.32, hspace=0.35
)

ax_a = fig.add_subplot(gs[0, 0])
ax_b = fig.add_subplot(gs[0, 1])
ax_c = fig.add_subplot(gs[1, 0])
ax_d = fig.add_subplot(gs[1, 1])

# (a)
sns.heatmap(
    static_plot_df,
    ax=ax_a,
    annot=True,
    fmt=".2f",
    cmap="mako",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Impact = range / baseline", "shrink": 0.92},
    annot_kws={"size": 9},
)
ax_a.set_title("(a) Static sensitivity heatmap", loc="left", fontsize=15, pad=10)
ax_a.set_xlabel("")
ax_a.set_ylabel("Parameter", fontsize=12)
ax_a.tick_params(axis="x", labelrotation=18, labelsize=9)
for t in ax_a.get_xticklabels():
    t.set_ha("right")
ax_a.tick_params(axis="y", labelsize=10)

# (b)
sns.barplot(
    data=rank_df,
    y="param_show",
    x=rank_col,
    ax=ax_b,
    orient="h",
    color=sns.color_palette("viridis", 6)[4],
)
ax_b.set_title(r"(b) Ranking by $R^2(\Psi,\mathrm{branches})$", loc="left", fontsize=15, pad=10)
ax_b.set_xlabel(r"Relative impact on $R^2(\Psi,\mathrm{branches})$", fontsize=12)
ax_b.set_ylabel("")
ax_b.tick_params(axis="y", labelsize=10)
ax_b.tick_params(axis="x", labelsize=10)
for container in ax_b.containers:
    ax_b.bar_label(container, fmt="%.2f", padding=3, fontsize=9)

# (c)
sns.heatmap(
    dynamic_plot_df,
    ax=ax_c,
    annot=True,
    fmt=".2f",
    cmap="rocket",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Impact = range / baseline", "shrink": 0.92},
    annot_kws={"size": 9},
)
ax_c.set_title("(c) Dynamic sensitivity heatmap", loc="left", fontsize=15, pad=10)
ax_c.set_xlabel("")
ax_c.set_ylabel("Parameter", fontsize=12)
ax_c.tick_params(axis="x", labelrotation=18, labelsize=9)
for t in ax_c.get_xticklabels():
    t.set_ha("right")
ax_c.tick_params(axis="y", labelsize=10)

# (d)
x = np.arange(len(panel_d_piv))
x_static = x - 0.11
x_dynamic = x + 0.11

for i, row in panel_d_piv.iterrows():
    ax_d.plot(
        [x_static[i], x_dynamic[i]],
        [row["static"], row["dynamic"]],
        lw=1.8, alpha=0.75, zorder=2
    )

ax_d.scatter(x_static, panel_d_piv["static"], s=80, marker="o", label="Static", zorder=3)
ax_d.scatter(x_dynamic, panel_d_piv["dynamic"], s=92, marker="D", label="Dynamic", zorder=3)

y_all = np.r_[panel_d_piv["static"].values, panel_d_piv["dynamic"].values]
ymin, ymax = np.nanmin(y_all), np.nanmax(y_all)
yr = ymax - ymin if np.isfinite(ymax - ymin) and (ymax > ymin) else 1.0
for i, row in panel_d_piv.iterrows():
    ax_d.text(
        x_dynamic[i] + 0.03,
        row["dynamic"] + 0.03 * yr,
        f"{row['delta_dyn_minus_static']:+.2f}",
        fontsize=8.5,
        ha="left",
        va="bottom",
    )

ax_d.set_xticks(x)
ax_d.set_xticklabels(panel_d_piv["label_show"], fontsize=9)
ax_d.set_ylabel(_pretty_metric_name(PANEL_D_METRIC), fontsize=12)
ax_d.set_title("(d) Static vs dynamic robustness check", loc="left", fontsize=15, pad=10)
ax_d.legend(frameon=True, fontsize=9, loc="best")

summary_text = (
    rf"$\rho_s$ = {rho_s:.2f}" if np.isfinite(rho_s) else r"$\rho_s$ = NA"
) + "\n" + (
    rf"$r$ = {pearson_r:.2f}" if np.isfinite(pearson_r) else r"$r$ = NA"
) + "\n" + (
    rf"mean |Δ| = {mean_abs_shift:.2f}" if np.isfinite(mean_abs_shift) else r"mean |Δ| = NA"
)
ax_d.text(
    0.98, 0.03, summary_text,
    transform=ax_d.transAxes,
    ha="right", va="bottom", fontsize=9,
    bbox=dict(boxstyle="round,pad=0.28", facecolor="white", alpha=0.92, edgecolor="0.7")
)

fig.suptitle(
    "Figure 6. Sensitivity analysis and dynamic robustness check",
    fontsize=21, fontweight="bold", y=0.98
)
plt.tight_layout(rect=[0, 0, 1, 0.965])

fig_png, fig_pdf = _safe_save(
    fig,
    PAPER_DIR / "Figure6_sensitivity_dynamic_revised.png",
    PAPER_DIR / "Figure6_sensitivity_dynamic_revised.pdf",
)
plt.show()

print("Saved to:")
print(fig_png)
print(fig_pdf)
print(PAPER_DIR / "Figure6_panel_d_points_revised.csv")


In [ ]:

# ============================================================
# PATCH — Supplementary figures (focus on S1) + robust saving
# 运行方式：放在原 Supplementary cell 后面运行；它会覆盖旧定义
# 然后执行：make_S1(); make_S2(); make_S3()
# ============================================================

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

required_vars = ["OUT_DIR", "RUN_PRESET"]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"缺少这些变量：{missing}")

# 用更短的目录，避免 Windows 路径/保存问题
SUPP_DIR = OUT_DIR / "supp_figs"
SUPP_DIR.mkdir(parents=True, exist_ok=True)

# S1：不再画 E 轴 formation curves，而是只画 active-zone 定义 z<=? 的趋势图
# 点数也比原来更多
SUPP_ACTIVE_ZMAX_LIST = None   # None = 自动生成更密的 z 点
SUPP_S1_N_Z = 7
SUPP_S1_M_LIST = [0.25, 0.75]  # 默认不用 0.90
SUPP_S1_STAGE = "crit"
SUPP_S1_STAT = "median"

sns.set_theme(style="whitegrid", context="talk", font_scale=0.92)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 350

def _sanitize_stem(stem):
    stem = str(stem)
    stem = re.sub(r'[<>:"/\\|?*]+', "_", stem)
    stem = stem.replace(" ", "_")
    return stem[:120]

def savefig_both(fig, stem):
    stem = _sanitize_stem(stem)
    png = SUPP_DIR / f"{stem}.png"
    pdf = SUPP_DIR / f"{stem}.pdf"
    png.parent.mkdir(parents=True, exist_ok=True)
    pdf.parent.mkdir(parents=True, exist_ok=True)

    try:
        fig.savefig(png, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white")
        _ok(stem, png)
    except Exception as e:
        short_dir = OUT_DIR / "fig_tmp"
        short_dir.mkdir(parents=True, exist_ok=True)
        png2 = short_dir / png.name
        pdf2 = short_dir / pdf.name
        print(f"[save fallback] {e}")
        fig.savefig(png2, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf2, bbox_inches="tight", facecolor="white")
        _ok(stem, png2)
    plt.close(fig)

def _carry_meta_columns(df_raw, out):
    preferred = [
        "zone_tag", "zmax", "z_frac", "label", "param", "level_name",
        "mode", "m_bucket", "tag"
    ]
    for c in preferred:
        if c in df_raw.columns and c not in out.columns:
            out[c] = df_raw[c]
    return out

def standardize_mE_summary(df_raw, fixed_T=700.0):
    out = pd.DataFrame(index=df_raw.index)
    out["source"] = "mE"
    out["m_target"] = df_raw[pick_col(df_raw, ["m_target", "m", "m_actual_mean", "m_actual"])]
    out["m_actual"] = df_raw[pick_col(df_raw, ["m_actual_mean", "m_actual", "m_target", "m"])]
    e_col = pick_col(df_raw, ["E", "E_vnm", "E_field", "Efield_vnm", "E_fixed", "E_fixed_vnm"])
    out["E"] = df_raw[e_col]
    t_col = pick_col(df_raw, ["T", "T_K", "temp_K", "temperature_K", "T_fixed", "T_fixed_K"], required=False)
    out["T"] = df_raw[t_col] if t_col is not None else fixed_T

    sigma_col = pick_col(df_raw, ["sigma_E_mean", "sigma_E", "barrier_std_z", "std_em_z"], required=False)
    out["sigma_E"] = df_raw[sigma_col] if sigma_col is not None else np.nan

    lc_col = pick_col(df_raw, ["lc_mean", "l_c", "lc", "l_c_mean"], required=False)
    out["lc"] = df_raw[lc_col] if lc_col is not None else np.nan

    delta_col = pick_col(df_raw, ["Delta_mean", "Delta"], required=False)
    if delta_col is not None:
        out["Delta"] = df_raw[delta_col]
    else:
        out["Delta"] = out["sigma_E"] / (KB_EV * out["T"])

    formed_col = pick_col(df_raw, ["formed_prob", "Pform", "formation_prob"])
    out["formed_prob"] = df_raw[formed_col]

    tset_col = pick_col(df_raw, ["log10_t_set_mean", "log10_t_set", "t_set_log10_mean"], required=False)
    out["log10_t_set"] = df_raw[tset_col] if tset_col is not None else np.nan

    out = ensure_numeric(out, ["m_target", "m_actual", "E", "T", "sigma_E", "lc", "Delta", "formed_prob", "log10_t_set"])
    out = _carry_meta_columns(df_raw, out)
    return out

def standardize_mE_topology(df_raw, stage="crit", stat="median", fixed_T=700.0):
    out = pd.DataFrame(index=df_raw.index)
    out["source"] = "mE"
    out["m_target"] = df_raw[pick_col(df_raw, ["m_target", "m", "m_actual_mean", "m_actual"])]
    out["m_actual"] = df_raw[pick_col(df_raw, ["m_actual_mean", "m_actual", "m_target", "m"])]
    e_col = pick_col(df_raw, ["E", "E_vnm", "E_field", "Efield_vnm", "E_fixed", "E_fixed_vnm"])
    out["E"] = df_raw[e_col]
    t_col = pick_col(df_raw, ["T", "T_K", "temp_K", "temperature_K", "T_fixed", "T_fixed_K"], required=False)
    out["T"] = df_raw[t_col] if t_col is not None else fixed_T

    sigma_col = pick_col(df_raw, ["sigma_E_mean", "sigma_E", "barrier_std_z", "std_em_z", "sigma_real_mean"], required=False)
    out["sigma_E"] = df_raw[sigma_col] if sigma_col is not None else np.nan

    lc_col = pick_col(df_raw, ["lc_mean", "l_c", "lc", "l_c_mean"], required=False)
    out["lc"] = df_raw[lc_col] if lc_col is not None else np.nan

    delta_col = pick_col(df_raw, ["Delta_mean", "Delta"], required=False)
    if delta_col is not None:
        out["Delta"] = df_raw[delta_col]
    else:
        out["Delta"] = out["sigma_E"] / (KB_EV * out["T"])

    pform_col = pick_col(df_raw, ["formed_prob", "Pform", "formation_prob"], required=False)
    out["formed_prob"] = df_raw[pform_col] if pform_col is not None else np.nan

    stage = stage.lower()
    stat = stat.lower()

    branch_candidates = ["branches_topo_main"]
    tort_candidates = ["tortuosity_topo_main"]
    neck_candidates = ["neck_topo_main"]

    if stage == "crit":
        if stat == "median":
            branch_candidates += ["branches_crit_median_formed"]
            tort_candidates += ["tortuosity_nm_crit_median_formed"]
            neck_candidates += ["neck_nm_crit_median_formed"]
        else:
            branch_candidates += ["branches_crit_mean_formed"]
            tort_candidates += ["tortuosity_nm_crit_mean_formed"]
            neck_candidates += ["neck_nm_crit_mean_formed"]
    elif stage == "lrs":
        if stat == "median":
            branch_candidates += ["branches_lrs_median_formed", "branches_median_formed"]
            tort_candidates += ["tortuosity_nm_lrs_median_formed", "tortuosity_nm_median_formed"]
            neck_candidates += ["neck_nm_lrs_median_formed", "neck_nm_median_formed"]
        else:
            branch_candidates += ["branches_lrs_mean_formed", "branches_mean_formed"]
            tort_candidates += ["tortuosity_nm_lrs_mean_formed", "tortuosity_nm_mean_formed"]
            neck_candidates += ["neck_nm_lrs_mean_formed", "neck_nm_mean_formed"]

    out["branches"] = df_raw[pick_col(df_raw, branch_candidates)]
    out["tortuosity"] = df_raw[pick_col(df_raw, tort_candidates)]
    out["neck_nm"] = df_raw[pick_col(df_raw, neck_candidates)]

    out = ensure_numeric(out, ["m_target", "m_actual", "E", "T", "sigma_E", "lc", "Delta", "formed_prob", "branches", "tortuosity", "neck_nm"])
    out = _carry_meta_columns(df_raw, out)
    return out

def _build_active_z_list(z_ox_end, n=7):
    z_lo = max(7, int(round(0.24 * z_ox_end)))
    z_hi = min(z_ox_end - 2, int(round(0.62 * z_ox_end)))
    z_vals = np.linspace(z_lo, z_hi, n)
    z_vals = np.unique(np.round(z_vals).astype(int))
    return [int(z) for z in z_vals if 2 <= z < z_ox_end]

def _run_s1_dataset():
    if not SUPP_RUN_EXPENSIVE:
        raise RuntimeError("SUPP_RUN_EXPENSIVE=False，S1 需要打开这个开关。")

    base_tmp = load_phase2_ns(save_subdir="_supp_phase2_probe")
    z_ox_end = int(base_tmp["Z_OX_END"])

    if SUPP_ACTIVE_ZMAX_LIST is None:
        z_list = _build_active_z_list(z_ox_end, n=SUPP_S1_N_Z)
    else:
        z_list = [int(z) for z in SUPP_ACTIVE_ZMAX_LIST if int(z) < z_ox_end]

    rows = []
    topo_rows = []

    for zmax in z_list:
        frac = float(zmax) / float(z_ox_end)
        ns = load_phase2_ns(
            extra_overrides={"ACTIVE_Z_MAX_FRAC": frac},
            save_subdir=f"_supp_s1_z{zmax}",
        )
        print(f"[supp:S1] running active-zone z<={zmax} (frac={frac:.4f})")

        df = ns["run_phase2_grid"](
            mode="mE",
            m_list=SUPP_S1_M_LIST,
            E_list=SUPP_ME_E_LIST,
            T_fixed=SUPP_ME_T_FIXED,
            n_structure_seeds=SUPP_N_STRUCTURE,
            n_mc_per_structure=SUPP_N_MC,
            n_jobs=SUPP_N_JOBS,
        )

        summ = ns["summarize_phase2"](df, mode="mE")
        summ["zone_tag"] = f"z<={zmax}"
        summ["zmax"] = int(zmax)
        summ["z_frac"] = frac
        rows.append(summ)

        topo = standardize_mE_topology(summ, stage=SUPP_S1_STAGE, stat=SUPP_S1_STAT, fixed_T=SUPP_ME_T_FIXED)
        topo["zone_tag"] = f"z<={zmax}"
        topo["zmax"] = int(zmax)
        topo["z_frac"] = frac
        topo_rows.append(topo)

    all_df = pd.concat(rows, ignore_index=True)
    all_topo = pd.concat(topo_rows, ignore_index=True)

    std_sum = standardize_mE_summary(all_df, fixed_T=SUPP_ME_T_FIXED)

    e50_rows = []
    for (zmax, zone_tag), g0 in std_sum.groupby(["zmax", "zone_tag"]):
        e50_tmp, _ = fit_e50_by_m(g0)
        if len(e50_tmp):
            e50_tmp["zmax"] = int(zmax)
            e50_tmp["zone_tag"] = zone_tag
            e50_rows.append(e50_tmp)
    e50_all = pd.concat(e50_rows, ignore_index=True) if len(e50_rows) else pd.DataFrame()

    near_rows = []
    if len(e50_all):
        for (zmax, zone_tag), g in all_topo.groupby(["zmax", "zone_tag"]):
            for m0 in sorted(pd.unique(g["m_target"].dropna())):
                gg = g[np.isclose(g["m_target"], float(m0))].copy()
                e50_sub = e50_all[(e50_all["zmax"] == zmax) & (e50_all["zone_tag"] == zone_tag) & (np.isclose(e50_all["m_target"], float(m0)))]
                if len(gg) == 0 or len(e50_sub) == 0:
                    continue
                e50v = float(e50_sub.iloc[0]["E50"])
                if not np.isfinite(e50v):
                    continue
                gg["dist"] = np.abs(gg["E"] - e50v)
                row = gg.sort_values("dist").iloc[0]
                near_rows.append({
                    "zmax": int(zmax),
                    "zone_tag": zone_tag,
                    "m_target": float(m0),
                    "E50": e50v,
                    "wE": float(e50_sub.iloc[0]["wE"]) if "wE" in e50_sub.columns else np.nan,
                    "branches": row.get("branches", np.nan),
                    "tortuosity": row.get("tortuosity", np.nan),
                    "formed_prob": row.get("formed_prob", np.nan),
                })
    near_df = pd.DataFrame(near_rows)
    return std_sum, e50_all, near_df, z_list

def make_S1():
    std_sum, e50_all, near_df, z_list = _run_s1_dataset()

    if len(e50_all) == 0:
        _skip("Figure_S1", "没有可用的 E50 拟合结果")
        return

    preferred_m = [m for m in SUPP_S1_M_LIST if np.any(np.isclose(e50_all["m_target"], m))]
    if len(preferred_m) == 0:
        preferred_m = sorted(pd.unique(e50_all["m_target"].dropna()))
    if len(preferred_m) == 0:
        _skip("Figure_S1", "没有可用的 m_target")
        return

    palette = sns.color_palette("viridis", n_colors=len(preferred_m))
    color_map = {m: palette[i] for i, m in enumerate(preferred_m)}

    fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.8))
    axes = axes.ravel()

    # (a) E50 vs zmax
    ax = axes[0]
    sub = e50_all[e50_all["m_target"].isin(preferred_m)].copy()
    for m0 in preferred_m:
        g = sub[np.isclose(sub["m_target"], m0)].sort_values("zmax")
        if len(g) == 0:
            continue
        ax.plot(g["zmax"], g["E50"], marker="o", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
    ax.set_title("(a) Threshold field $E_{50}$ vs active-zone definition", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_ylabel(r"$E_{50}$ (V/nm)")
    ax.set_xticks(z_list)
    ax.legend(title=None, frameon=True)

    # (b) transition width vs zmax
    ax = axes[1]
    sub = e50_all[e50_all["m_target"].isin(preferred_m)].copy()
    for m0 in preferred_m:
        g = sub[np.isclose(sub["m_target"], m0)].sort_values("zmax")
        if len(g) == 0:
            continue
        ax.plot(g["zmax"], g["wE"], marker="s", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
    ax.set_title("(b) Transition width vs active-zone definition", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_ylabel(r"$w_E$ (V/nm)")
    ax.set_xticks(z_list)
    ax.legend(title=None, frameon=True)

    # (c) branches near E50
    ax = axes[2]
    if len(near_df):
        sub = near_df[near_df["m_target"].isin(preferred_m)].copy()
        for m0 in preferred_m:
            g = sub[np.isclose(sub["m_target"], m0)].sort_values("zmax")
            if len(g) == 0:
                continue
            ax.plot(g["zmax"], g["branches"], marker="o", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
        ax.set_ylabel("Branches near $E_{50}$")
        ax.legend(title=None, frameon=True)
    else:
        ax.text(0.5, 0.5, "No valid rows near $E_{50}$", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title("(c) Branches near fitted $E_{50}$", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_xticks(z_list)

    # (d) tortuosity near E50
    ax = axes[3]
    if len(near_df):
        sub = near_df[near_df["m_target"].isin(preferred_m)].copy()
        for m0 in preferred_m:
            g = sub[np.isclose(sub["m_target"], m0)].sort_values("zmax")
            if len(g) == 0:
                continue
            ax.plot(g["zmax"], g["tortuosity"], marker="D", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
        ax.set_ylabel("Tortuosity near $E_{50}$")
        ax.legend(title=None, frameon=True)
    else:
        ax.text(0.5, 0.5, "No valid rows near $E_{50}$", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title("(d) Tortuosity near fitted $E_{50}$", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_xticks(z_list)

    fig.suptitle("Figure S1. Sensitivity to active-zone depth definition", fontsize=20, fontweight="bold", y=0.985)
    plt.tight_layout(rect=[0, 0, 1, 0.965])
    savefig_both(fig, "Figure_S1_active_zone_depth_revised")

    # 导出表格，写正文和 caption 时更方便
    e50_all.to_csv(SUPP_DIR / "Figure_S1_E50_table.csv", index=False)
    if len(near_df):
        near_df.to_csv(SUPP_DIR / "Figure_S1_nearE50_table.csv", index=False)

print("Supplementary patch loaded. Now run: make_S1(); make_S2(); make_S3()")


In [ ]:

# ============================================================
# PATCH v3 — Figure 6(d) auto metric + robust S1 output
# 运行方式：
# 1) 跑完前面的主 notebook
# 2) 再运行这个 cell
# 3) 然后执行：
#       render_figure6_v3()
#       make_S1_v3()
# ============================================================

from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

required_vars = [
    "OUT_DIR",
    "static_impact_df",
    "dynamic_impact_df",
    "dynamic_raw_df",
    "summarize_runs",
    "load_phase2_ns",
    "standardize_mE_summary",
    "standardize_mE_topology",
    "pick_col",
    "ensure_numeric",
    "KB_EV",
    "SUPP_ME_E_LIST",
    "SUPP_ME_T_FIXED",
    "SUPP_N_STRUCTURE",
    "SUPP_N_MC",
    "SUPP_N_JOBS",
]
missing = [v for v in required_vars if v not in globals()]
if missing:
    raise RuntimeError(f"缺少这些变量，说明前面的单元还没跑完：{missing}")

PAPER_DIR_V3 = OUT_DIR / "paper_figures_v3"
PAPER_DIR_V3.mkdir(parents=True, exist_ok=True)

SUPP_DIR_V3 = OUT_DIR / "supplementary_figures_v3"
SUPP_DIR_V3.mkdir(parents=True, exist_ok=True)

# 也同步保存到原来的目录名，避免你“看起来像没输出”
SUPP_DIR_COMPAT = OUT_DIR / "supplementary_figures_from_code"
SUPP_DIR_COMPAT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="talk", font_scale=0.92)
plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.labelweight"] = "bold"
plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 350

# ---------- shared helpers ----------
def _sanitize_stem_v3(stem):
    stem = str(stem)
    stem = re.sub(r'[<>:"/\\|?*]+', "_", stem)
    stem = stem.replace(" ", "_")
    return stem[:120]

def savefig_both_v3(fig, stem, out_dir_primary=SUPP_DIR_V3, out_dir_extra=SUPP_DIR_COMPAT):
    stem = _sanitize_stem_v3(stem)
    out_dir_primary = Path(out_dir_primary)
    out_dir_extra = Path(out_dir_extra)
    out_dir_primary.mkdir(parents=True, exist_ok=True)
    out_dir_extra.mkdir(parents=True, exist_ok=True)

    png = out_dir_primary / f"{stem}.png"
    pdf = out_dir_primary / f"{stem}.pdf"
    png2 = out_dir_extra / f"{stem}.png"
    pdf2 = out_dir_extra / f"{stem}.pdf"

    saved = []
    try:
        fig.savefig(png, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white")
        saved.extend([png, pdf])
    except Exception as e:
        short_dir = OUT_DIR / "fig_tmp_v3"
        short_dir.mkdir(parents=True, exist_ok=True)
        png = short_dir / png.name
        pdf = short_dir / pdf.name
        fig.savefig(png, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf, bbox_inches="tight", facecolor="white")
        saved.extend([png, pdf])
        print(f"[save fallback] primary failed: {e}")

    # 再存一份到兼容目录
    try:
        fig.savefig(png2, dpi=350, bbox_inches="tight", facecolor="white")
        fig.savefig(pdf2, bbox_inches="tight", facecolor="white")
        saved.extend([png2, pdf2])
    except Exception as e:
        print(f"[save compat skipped] {e}")

    print("[saved]")
    for p in saved:
        print("  ", p)
    plt.close(fig)
    return saved

def _pretty_point_label_v3(lbl):
    lbl = str(lbl)
    lbl = lbl.replace("m000", "m=0.00")
    lbl = lbl.replace("m025", "m=0.25")
    lbl = lbl.replace("m050", "m=0.50")
    lbl = lbl.replace("m075", "m=0.75")
    lbl = lbl.replace("m090", "m=0.90")
    lbl = lbl.replace("_sub", "\nsub")
    lbl = lbl.replace("_thr", "\nthr")
    lbl = lbl.replace("_sup", "\nsup")
    return lbl

def _pretty_metric_name_v3(x):
    mp = {
        "formed_prob": "Formation probability",
        "m_final_mean": r"Final $m$",
        "Delta_final_mean": r"Final $\Delta$",
        "log10_t_set_median_formed": r"$\log_{10}(t_{\mathrm{set}})$",
        "branches_crit_median_formed": "Branches at critical state",
        "tortuosity_nm_crit_median_formed": "Tortuosity at critical state (nm)",
        "formed_prob_mean6": "Formation\nprobability",
        "log10_t_set_med_mean6": r"$\log_{10}(t_{\mathrm{set}})$",
        "branches_crit_med_mean6": "Branches at\ncritical state",
        "tortuosity_crit_med_mean6": "Tortuosity at\ncritical state",
        "E50_m025": r"$E_{50}$ at $m=0.25$",
        "R2_Psi_branches": r"$R^2(\Psi,\mathrm{branches})$",
        "abs_delta_formed_prob_mean6": r"$|\Delta P_{\mathrm{form}}|$",
        "abs_delta_log10_t_set_median_formed_mean6": r"$|\Delta \log_{10}(t_{\mathrm{set}})|$",
        "abs_delta_branches_crit_median_formed_mean6": r"$|\Delta \mathrm{Branches}_{\mathrm{crit}}|$",
        "delta_m_final_mean_mean6": r"$\Delta m_{\mathrm{final}}$",
        "delta_Delta_final_mean_mean6": r"$\Delta \Delta_{\mathrm{final}}$",
    }
    return mp.get(x, x)

def _build_static_dynamic_pairs(metric, m_keep=(0.25,0.50,0.75), exclude_m=(0.90,)):
    base_dyn = dynamic_raw_df.copy()
    if "param" in base_dyn.columns:
        base_dyn = base_dyn[base_dyn["param"] == "BASELINE"].copy()
    if "level_name" in base_dyn.columns:
        base_dyn = base_dyn[base_dyn["level_name"] == "base"].copy()

    point_summary = summarize_runs(
        base_dyn,
        group_cols=("label", "mode", "m_target", "E", "T")
    ).copy()

    if metric not in point_summary.columns:
        return pd.DataFrame(), point_summary

    piv = (
        point_summary
        .pivot_table(
            index=["label", "m_target", "E", "T"],
            columns="mode",
            values=metric,
            aggfunc="first"
        )
        .reset_index()
    )
    if ("static" not in piv.columns) or ("dynamic" not in piv.columns):
        return pd.DataFrame(), point_summary

    piv = piv.dropna(subset=["static", "dynamic"]).copy()
    if len(piv) == 0:
        return piv, point_summary

    # 先保留中低 m；如果没有足够点，再放宽
    m_keep = tuple(m_keep) if m_keep is not None else tuple()
    if len(m_keep):
        piv0 = piv[piv["m_target"].isin(m_keep)].copy()
        if len(piv0) >= 2:
            piv = piv0

    if exclude_m:
        piv0 = piv[~piv["m_target"].isin(tuple(exclude_m))].copy()
        if len(piv0) >= 2:
            piv = piv0

    piv["label_show"] = piv["label"].astype(str).map(_pretty_point_label_v3)
    piv["delta_dyn_minus_static"] = piv["dynamic"] - piv["static"]
    piv = piv.sort_values(["m_target", "E", "label_show"]).reset_index(drop=True)
    return piv, point_summary

def _auto_select_panel_d_metric(
    preferred=None,
    m_keep=(0.25,0.50,0.75),
    exclude_m=(0.90,),
    min_pairs=2,
    tol=1e-8,
):
    if preferred is None:
        preferred = [
            "branches_crit_median_formed",
            "tortuosity_nm_crit_median_formed",
            "log10_t_set_median_formed",
            "m_final_mean",
            "Delta_final_mean",
            "formed_prob",
        ]

    best = None
    best_score = -np.inf
    best_piv = None

    for metric in preferred:
        piv, point_summary = _build_static_dynamic_pairs(metric, m_keep=m_keep, exclude_m=exclude_m)
        if len(piv) < min_pairs:
            continue
        diff = np.asarray(piv["delta_dyn_minus_static"], dtype=float)
        static = np.asarray(piv["static"], dtype=float)
        dynamic = np.asarray(piv["dynamic"], dtype=float)
        if not np.any(np.isfinite(diff)):
            continue

        max_abs = np.nanmax(np.abs(diff))
        spread = np.nanstd(np.r_[static, dynamic])
        n_nonzero = int(np.sum(np.abs(diff) > tol))
        # 优先选择“确实有变化”的指标
        if (max_abs <= tol) and (spread <= tol):
            continue

        score = (
            3.0 * n_nonzero +
            1.5 * float(np.nanmean(np.abs(diff))) +
            0.8 * float(np.nanstd(diff)) +
            0.1 * len(piv)
        )
        if score > best_score:
            best_score = score
            best = metric
            best_piv = piv.copy()

    if best is None:
        # 最后兜底
        for metric in preferred:
            piv, _ = _build_static_dynamic_pairs(metric, m_keep=None, exclude_m=None)
            if len(piv) >= min_pairs:
                best = metric
                best_piv = piv.copy()
                break

    if best is None:
        raise RuntimeError("panel (d) 没有找到可用且有变化的 static/dynamic 指标。")
    return best, best_piv

def render_figure6_v3():
    # a/b/c 仍沿用你前面已经算好的 impact 表
    static_cols = [
        "formed_prob_mean6",
        "log10_t_set_med_mean6",
        "branches_crit_med_mean6",
        "tortuosity_crit_med_mean6",
        "E50_m025",
        "R2_Psi_branches",
    ]
    dynamic_cols = [
        "abs_delta_formed_prob_mean6",
        "abs_delta_log10_t_set_median_formed_mean6",
        "abs_delta_branches_crit_median_formed_mean6",
        "delta_m_final_mean_mean6",
        "delta_Delta_final_mean_mean6",
    ]

    static_plot_df = static_impact_df.copy()
    static_plot_df = static_plot_df[static_plot_df["param"] != "BASELINE"].copy()
    static_cols_use = [c for c in static_cols if c in static_plot_df.columns]
    static_plot_df = static_plot_df.set_index("param")[static_cols_use].copy()

    dynamic_plot_df = dynamic_impact_df.copy()
    dynamic_plot_df = dynamic_plot_df[dynamic_plot_df["param"] != "BASELINE"].copy()
    dynamic_cols_use = [c for c in dynamic_cols if c in dynamic_plot_df.columns]
    dynamic_plot_df = dynamic_plot_df.set_index("param")[dynamic_cols_use].copy()

    row_map = {
        "ACTIVE_Z_MAX_FRAC": r"$z_{\mathrm{active}}^{\max}$",
        "ALPHA_FIELD": r"$\alpha_E$",
        "LC0_CELLS": r"$l_{c0}$",
        "LC_BETA_CELLS": r"$\beta_{lc}$",
        "MU_X_EXTRA_EV": r"$\mu_x^{\mathrm{extra}}$",
        "P_SIGMA": r"$p_0$",
        "SIGMA_DIS_EV": r"$\sigma_{\mathrm{dis}}$",
        "SIGMA_RES_EV": r"$\sigma_{\mathrm{res}}$",
        "DYN_ALPHA_SIGMA": r"Dynamic $\alpha_\sigma$",
        "DYN_M_COUNT_MODE": "Dynamic m-count\nmode",
        "DYN_UPDATE_EVENTS": "Dynamic update\ninterval",
    }
    col_map = {
        "formed_prob_mean6": "Formation\nprobability",
        "log10_t_set_med_mean6": r"$\log_{10}(t_{\mathrm{set}})$",
        "branches_crit_med_mean6": "Branches at\ncritical state",
        "tortuosity_crit_med_mean6": "Tortuosity at\ncritical state",
        "E50_m025": r"$E_{50}$ at $m=0.25$",
        "R2_Psi_branches": r"$R^2(\Psi,\mathrm{branches})$",
        "abs_delta_formed_prob_mean6": r"$|\Delta P_{\mathrm{form}}|$",
        "abs_delta_log10_t_set_median_formed_mean6": r"$|\Delta \log_{10}(t_{\mathrm{set}})|$",
        "abs_delta_branches_crit_median_formed_mean6": r"$|\Delta \mathrm{Branches}_{\mathrm{crit}}|$",
        "delta_m_final_mean_mean6": r"$\Delta m_{\mathrm{final}}$",
        "delta_Delta_final_mean_mean6": r"$\Delta \Delta_{\mathrm{final}}$",
    }
    static_plot_df.index = [row_map.get(x, x) for x in static_plot_df.index]
    static_plot_df.columns = [col_map.get(x, x) for x in static_plot_df.columns]
    dynamic_plot_df.index = [row_map.get(x, x) for x in dynamic_plot_df.index]
    dynamic_plot_df.columns = [col_map.get(x, x) for x in dynamic_plot_df.columns]

    rank_col = "R2_Psi_branches"
    if rank_col not in static_impact_df.columns:
        raise RuntimeError("static_impact_df 里没有 R2_Psi_branches，panel (b) 无法画。")
    rank_df = static_impact_df.copy()
    rank_df = rank_df[rank_df["param"] != "BASELINE"].copy()
    rank_df["param_show"] = rank_df["param"].map(row_map).fillna(rank_df["param"])
    rank_df = rank_df.sort_values(rank_col, ascending=False)

    # panel d 自动挑“确实有变化”的指标
    chosen_metric, panel_d_piv = _auto_select_panel_d_metric(
        preferred=[
            "branches_crit_median_formed",
            "tortuosity_nm_crit_median_formed",
            "log10_t_set_median_formed",
            "m_final_mean",
            "Delta_final_mean",
            "formed_prob",
        ],
        m_keep=(0.25, 0.50, 0.75),
        exclude_m=(0.90,),
    )

    panel_d_piv.to_csv(PAPER_DIR_V3 / "Figure6_panel_d_points_v3.csv", index=False)

    valid_corr = panel_d_piv[["static", "dynamic"]].dropna()
    if len(valid_corr) >= 2:
        rho_s = valid_corr["static"].rank().corr(valid_corr["dynamic"].rank())
        pearson_r = valid_corr["static"].corr(valid_corr["dynamic"])
    else:
        rho_s = np.nan
        pearson_r = np.nan
    mean_abs_shift = np.nanmean(np.abs(panel_d_piv["delta_dyn_minus_static"].values))

    fig = plt.figure(figsize=(17, 10), constrained_layout=False)
    gs = fig.add_gridspec(
        nrows=2, ncols=2,
        width_ratios=[1.15, 1.0],
        height_ratios=[1.0, 1.0],
        wspace=0.32, hspace=0.35
    )
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 0])
    ax_d = fig.add_subplot(gs[1, 1])

    sns.heatmap(
        static_plot_df,
        ax=ax_a,
        annot=True, fmt=".2f", cmap="mako",
        linewidths=0.8, linecolor="white",
        cbar_kws={"label": "Impact = range / baseline", "shrink": 0.92},
        annot_kws={"size": 9},
    )
    ax_a.set_title("(a) Static sensitivity heatmap", loc="left", fontsize=15, pad=10)
    ax_a.set_xlabel("")
    ax_a.set_ylabel("Parameter", fontsize=12)
    ax_a.tick_params(axis="x", labelrotation=18, labelsize=9)
    for t in ax_a.get_xticklabels():
        t.set_ha("right")
    ax_a.tick_params(axis="y", labelsize=10)

    sns.barplot(
        data=rank_df,
        y="param_show",
        x=rank_col,
        ax=ax_b,
        orient="h",
        color=sns.color_palette("viridis", 6)[4],
    )
    ax_b.set_title(r"(b) Ranking by $R^2(\Psi,\mathrm{branches})$", loc="left", fontsize=15, pad=10)
    ax_b.set_xlabel(r"Relative impact on $R^2(\Psi,\mathrm{branches})$", fontsize=12)
    ax_b.set_ylabel("")
    ax_b.tick_params(axis="y", labelsize=10)
    ax_b.tick_params(axis="x", labelsize=10)
    for container in ax_b.containers:
        ax_b.bar_label(container, fmt="%.2f", padding=3, fontsize=9)

    sns.heatmap(
        dynamic_plot_df,
        ax=ax_c,
        annot=True, fmt=".2f", cmap="rocket",
        linewidths=0.8, linecolor="white",
        cbar_kws={"label": "Impact = range / baseline", "shrink": 0.92},
        annot_kws={"size": 9},
    )
    ax_c.set_title("(c) Dynamic sensitivity heatmap", loc="left", fontsize=15, pad=10)
    ax_c.set_xlabel("")
    ax_c.set_ylabel("Parameter", fontsize=12)
    ax_c.tick_params(axis="x", labelrotation=18, labelsize=9)
    for t in ax_c.get_xticklabels():
        t.set_ha("right")
    ax_c.tick_params(axis="y", labelsize=10)

    x = np.arange(len(panel_d_piv))
    x_static = x - 0.11
    x_dynamic = x + 0.11

    for i, row in panel_d_piv.iterrows():
        ax_d.plot(
            [x_static[i], x_dynamic[i]],
            [row["static"], row["dynamic"]],
            lw=1.8, alpha=0.75, zorder=2
        )
    ax_d.scatter(x_static, panel_d_piv["static"], s=80, marker="o", label="Static", zorder=3)
    ax_d.scatter(x_dynamic, panel_d_piv["dynamic"], s=92, marker="D", label="Dynamic", zorder=3)

    y_all = np.r_[panel_d_piv["static"].values, panel_d_piv["dynamic"].values]
    ymin, ymax = np.nanmin(y_all), np.nanmax(y_all)
    yr = ymax - ymin if np.isfinite(ymax - ymin) and (ymax > ymin) else 1.0
    ax_d.set_ylim(ymin - 0.18*yr, ymax + 0.22*yr)

    for i, row in panel_d_piv.iterrows():
        ax_d.text(
            x_dynamic[i] + 0.03,
            row["dynamic"] + 0.04 * yr,
            f"{row['delta_dyn_minus_static']:+.2f}",
            fontsize=8.5,
            ha="left",
            va="bottom",
        )

    ax_d.set_xticks(x)
    ax_d.set_xticklabels(panel_d_piv["label_show"], fontsize=9)
    ax_d.set_ylabel(_pretty_metric_name_v3(chosen_metric), fontsize=12)
    ax_d.set_title(f"(d) Static vs dynamic: {_pretty_metric_name_v3(chosen_metric)}", loc="left", fontsize=15, pad=10)
    ax_d.legend(frameon=True, fontsize=9, loc="best")

    summary_text = (
        rf"$\rho_s$ = {rho_s:.2f}" if np.isfinite(rho_s) else r"$\rho_s$ = NA"
    ) + "\n" + (
        rf"$r$ = {pearson_r:.2f}" if np.isfinite(pearson_r) else r"$r$ = NA"
    ) + "\n" + (
        rf"mean |Δ| = {mean_abs_shift:.2f}" if np.isfinite(mean_abs_shift) else r"mean |Δ| = NA"
    )
    ax_d.text(
        0.98, 0.03, summary_text,
        transform=ax_d.transAxes,
        ha="right", va="bottom", fontsize=9,
        bbox=dict(boxstyle="round,pad=0.28", facecolor="white", alpha=0.92, edgecolor="0.7")
    )

    fig.suptitle(
        "Figure 6. Sensitivity analysis and dynamic robustness check",
        fontsize=21, fontweight="bold", y=0.98
    )
    plt.tight_layout(rect=[0, 0, 1, 0.965])

    paths = savefig_both_v3(fig, "Figure6_sensitivity_dynamic_v3", out_dir_primary=PAPER_DIR_V3, out_dir_extra=PAPER_DIR_V3)
    panel_d_piv.to_csv(PAPER_DIR_V3 / "Figure6_panel_d_points_v3.csv", index=False)
    print("[panel d metric]", chosen_metric)
    print(PAPER_DIR_V3 / "Figure6_panel_d_points_v3.csv")
    return chosen_metric, panel_d_piv

# ---------- S1 ----------
SUPP_S1_V3_N_Z = 9
SUPP_S1_V3_M_LIST = [0.25, 0.50, 0.75]
SUPP_ACTIVE_ZMAX_LIST_V3 = None  # None = 自动生成更密的 z 点
SUPP_S1_V3_STAGE = "crit"
SUPP_S1_V3_STAT = "median"

def _build_active_z_list_v3(z_ox_end, n=9):
    z_lo = max(6, int(round(0.20 * z_ox_end)))
    z_hi = min(z_ox_end - 2, int(round(0.68 * z_ox_end)))
    z_vals = np.linspace(z_lo, z_hi, n)
    z_vals = np.unique(np.round(z_vals).astype(int))
    return [int(z) for z in z_vals if 3 <= z < z_ox_end]

def _estimate_threshold_from_curve(E, P):
    E = np.asarray(E, dtype=float)
    P = np.asarray(P, dtype=float)
    mask = np.isfinite(E) & np.isfinite(P)
    E = E[mask]
    P = P[mask]
    if len(E) < 2:
        return np.nan, np.nan, "insufficient"

    order = np.argsort(E)
    E = E[order]
    P = P[order]

    # 去重
    df = pd.DataFrame({"E": E, "P": P}).groupby("E", as_index=False)["P"].mean()
    E = df["E"].values
    P = df["P"].values

    def interp_at(target):
        idx = np.where((P[:-1] - target) * (P[1:] - target) <= 0)[0]
        if len(idx):
            i = idx[0]
            x0, x1 = E[i], E[i+1]
            y0, y1 = P[i], P[i+1]
            if np.isclose(y1, y0):
                return 0.5 * (x0 + x1), "cross-flat"
            x = x0 + (target - y0) * (x1 - x0) / (y1 - y0)
            return float(x), "cross"
        # fallback: nearest point
        i = int(np.argmin(np.abs(P - target)))
        return float(E[i]), "nearest"

    e50, mode50 = interp_at(0.50)
    e25, _ = interp_at(0.25)
    e75, _ = interp_at(0.75)
    wE = abs(e75 - e25) if np.isfinite(e75) and np.isfinite(e25) else np.nan
    return e50, wE, mode50

def _run_s1_dataset_v3():
    if "SUPP_RUN_EXPENSIVE" in globals() and (not SUPP_RUN_EXPENSIVE):
        raise RuntimeError("SUPP_RUN_EXPENSIVE=False，S1 需要打开这个开关。")

    base_tmp = load_phase2_ns(save_subdir="_supp_phase2_probe_v3")
    z_ox_end = int(base_tmp["Z_OX_END"])

    if SUPP_ACTIVE_ZMAX_LIST_V3 is None:
        z_list = _build_active_z_list_v3(z_ox_end, n=SUPP_S1_V3_N_Z)
    else:
        z_list = [int(z) for z in SUPP_ACTIVE_ZMAX_LIST_V3 if int(z) < z_ox_end]

    rows = []
    topo_rows = []

    for zmax in z_list:
        frac = float(zmax) / float(z_ox_end)
        ns = load_phase2_ns(
            extra_overrides={"ACTIVE_Z_MAX_FRAC": frac},
            save_subdir=f"_supp_s1_v3_z{zmax}",
        )
        print(f"[S1-v3] running active-zone z<={zmax} (frac={frac:.4f})")

        df = ns["run_phase2_grid"](
            mode="mE",
            m_list=SUPP_S1_V3_M_LIST,
            E_list=SUPP_ME_E_LIST,
            T_fixed=SUPP_ME_T_FIXED,
            n_structure_seeds=SUPP_N_STRUCTURE,
            n_mc_per_structure=SUPP_N_MC,
            n_jobs=SUPP_N_JOBS,
        )

        summ = ns["summarize_phase2"](df, mode="mE")
        summ["zone_tag"] = f"z<={zmax}"
        summ["zmax"] = int(zmax)
        summ["z_frac"] = frac
        rows.append(summ)

        topo = standardize_mE_topology(summ, stage=SUPP_S1_V3_STAGE, stat=SUPP_S1_V3_STAT, fixed_T=SUPP_ME_T_FIXED)
        topo["zone_tag"] = f"z<={zmax}"
        topo["zmax"] = int(zmax)
        topo["z_frac"] = frac
        topo_rows.append(topo)

    all_df = pd.concat(rows, ignore_index=True)
    all_topo = pd.concat(topo_rows, ignore_index=True)
    std_sum = standardize_mE_summary(all_df, fixed_T=SUPP_ME_T_FIXED)

    e50_rows = []
    for (zmax, zone_tag, m0), g in std_sum.groupby(["zmax", "zone_tag", "m_target"]):
        g = g.sort_values("E")
        e50, wE, mode = _estimate_threshold_from_curve(g["E"].values, g["formed_prob"].values)
        e50_rows.append({
            "zmax": int(zmax),
            "zone_tag": zone_tag,
            "m_target": float(m0),
            "E50": e50,
            "wE": wE,
            "E50_mode": mode,
            "formed_prob_min": float(np.nanmin(g["formed_prob"].values)) if len(g) else np.nan,
            "formed_prob_max": float(np.nanmax(g["formed_prob"].values)) if len(g) else np.nan,
        })
    e50_all = pd.DataFrame(e50_rows)

    near_rows = []
    for (zmax, zone_tag, m0), g in all_topo.groupby(["zmax", "zone_tag", "m_target"]):
        sub = e50_all[
            (e50_all["zmax"] == int(zmax)) &
            (e50_all["zone_tag"] == zone_tag) &
            (np.isclose(e50_all["m_target"], float(m0)))
        ]
        if len(sub) == 0:
            continue
        e50v = float(sub.iloc[0]["E50"])
        if not np.isfinite(e50v):
            continue
        gg = g.copy()
        gg["dist"] = np.abs(gg["E"] - e50v)
        row = gg.sort_values("dist").iloc[0]
        near_rows.append({
            "zmax": int(zmax),
            "zone_tag": zone_tag,
            "m_target": float(m0),
            "E50": e50v,
            "wE": float(sub.iloc[0]["wE"]) if "wE" in sub.columns else np.nan,
            "E50_mode": sub.iloc[0]["E50_mode"],
            "branches": row.get("branches", np.nan),
            "tortuosity": row.get("tortuosity", np.nan),
            "neck_nm": row.get("neck_nm", np.nan),
            "formed_prob": row.get("formed_prob", np.nan),
        })
    near_df = pd.DataFrame(near_rows)
    return std_sum, e50_all, near_df, z_list

def make_S1_v3():
    std_sum, e50_all, near_df, z_list = _run_s1_dataset_v3()

    if len(e50_all) == 0:
        raise RuntimeError("S1-v3 没有生成任何 E50 估计结果。")

    preferred_m = [m for m in SUPP_S1_V3_M_LIST if np.any(np.isclose(e50_all["m_target"], m))]
    if len(preferred_m) == 0:
        preferred_m = sorted(pd.unique(e50_all["m_target"].dropna()))
    if len(preferred_m) == 0:
        raise RuntimeError("S1-v3 没有可用的 m_target。")

    palette = sns.color_palette("viridis", n_colors=len(preferred_m))
    color_map = {m: palette[i] for i, m in enumerate(preferred_m)}

    fig, axes = plt.subplots(2, 2, figsize=(14.8, 10.0))
    axes = axes.ravel()

    # (a) E50 vs zmax
    ax = axes[0]
    for m0 in preferred_m:
        g = e50_all[np.isclose(e50_all["m_target"], m0)].sort_values("zmax")
        if len(g) == 0:
            continue
        ax.plot(g["zmax"], g["E50"], marker="o", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
    ax.set_title("(a) Estimated threshold field vs active-zone definition", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_ylabel(r"Estimated $E_{50}$ (V/nm)")
    ax.set_xticks(z_list)
    ax.legend(title=None, frameon=True)

    # (b) transition width vs zmax
    ax = axes[1]
    for m0 in preferred_m:
        g = e50_all[np.isclose(e50_all["m_target"], m0)].sort_values("zmax")
        if len(g) == 0:
            continue
        ax.plot(g["zmax"], g["wE"], marker="s", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
    ax.set_title("(b) Transition width vs active-zone definition", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_ylabel(r"$w_E$ (V/nm)")
    ax.set_xticks(z_list)
    ax.legend(title=None, frameon=True)

    # (c) branches near E50
    ax = axes[2]
    if len(near_df):
        for m0 in preferred_m:
            g = near_df[np.isclose(near_df["m_target"], m0)].sort_values("zmax")
            if len(g) == 0:
                continue
            ax.plot(g["zmax"], g["branches"], marker="o", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
        ax.set_ylabel("Branches near estimated $E_{50}$")
        ax.legend(title=None, frameon=True)
    else:
        ax.text(0.5, 0.5, "No valid rows near estimated $E_{50}$", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title("(c) Branching response near the threshold", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_xticks(z_list)

    # (d) tortuosity near E50
    ax = axes[3]
    if len(near_df):
        for m0 in preferred_m:
            g = near_df[np.isclose(near_df["m_target"], m0)].sort_values("zmax")
            if len(g) == 0:
                continue
            ax.plot(g["zmax"], g["tortuosity"], marker="D", linewidth=2.2, ms=6, label=fr"$m={m0:.2f}$", color=color_map[m0])
        ax.set_ylabel("Tortuosity near estimated $E_{50}$")
        ax.legend(title=None, frameon=True)
    else:
        ax.text(0.5, 0.5, "No valid rows near estimated $E_{50}$", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
    ax.set_title("(d) Geometric complexity near the threshold", loc="left")
    ax.set_xlabel(r"Active-zone definition ($z \leq z_{\max}$)")
    ax.set_xticks(z_list)

    fig.suptitle("Figure S1. Sensitivity to active-zone depth definition", fontsize=20, fontweight="bold", y=0.985)
    plt.tight_layout(rect=[0, 0, 1, 0.965])
    savefig_both_v3(fig, "Figure_S1_active_zone_depth_v3", out_dir_primary=SUPP_DIR_V3, out_dir_extra=SUPP_DIR_COMPAT)

    e50_csv = SUPP_DIR_V3 / "Figure_S1_E50_table_v3.csv"
    near_csv = SUPP_DIR_V3 / "Figure_S1_nearE50_table_v3.csv"
    e50_all.to_csv(e50_csv, index=False)
    if len(near_df):
        near_df.to_csv(near_csv, index=False)
    print("[tables]")
    print("  ", e50_csv)
    if len(near_df):
        print("  ", near_csv)

    # 为了向下兼容，覆盖 make_S1
    return std_sum, e50_all, near_df, z_list

# 兼容：直接 make_S1() 也调用新版本
make_S1 = make_S1_v3

print("PATCH v3 loaded.")
print("Now run:")
print("  chosen_metric, panel_d_points = render_figure6_v3()")
print("  make_S1_v3()   # 或直接 make_S1()")
